# Spatial Trajectories PFC benchmark

Set STP_PROJECT before launching Jupyter. The bundle README lists inputs, kernels and run order. Run the sections for your workflow.

In [ ]:
import os
from pathlib import Path

# Set these before starting Jupyter. Existing per-notebook overrides still apply.
if "STP_PROJECT" not in os.environ:
    raise RuntimeError("Set STP_PROJECT to the experiment workspace before starting Jupyter.")
project_root = Path(os.environ["STP_PROJECT"]).expanduser().resolve()
os.environ["STP_PROJECT"] = str(project_root)
os.environ.setdefault("SPATIAL_TI_PROJECT", str(project_root))
os.environ.setdefault("PFC_PROJECT", str(project_root))
os.environ.setdefault("FLOW_STP_REPO", str(project_root))
os.environ.setdefault("QIAN_PFC_DATA_ROOT", str(project_root / "data/qian_pfc"))
os.environ.setdefault("QIAN_DATA_ROOT", os.environ["QIAN_PFC_DATA_ROOT"])
os.environ.setdefault("PFC_DATA_ROOT", os.environ["QIAN_PFC_DATA_ROOT"])
os.environ.setdefault("STVCR_REPO", str(project_root / "external/stVCR"))

## Render EN-ET-SP trajectories (Figure 6A)

In [ ]:
# Fresh kernel: NO S, maps, paths, selected, or earlier cell execution required.
# Requires saved moscot barycenters and the verified corrected stVCR exports.
# EXPLORATORY VISUALIZATION: EN-ET-SP target-territory PROXIMITY.
# Red means near any atlas-annotated EN-ET-SP cell (within fixed physical radius),
# NOT correct lineage/cell fate. Original 21-NN strict-majority scores archived
# as a diagnostic, alongside a predeclared radius sweep.
# No retraining, no target-dependent prediction changes. (Not individual lineage truth.)
# Reconstructs same five Fisher-transported Zoetrope contours from frozen nine-section model.
# No model retraining; no prediction-to-label registration or snapping.

import os, sys, json, hashlib, importlib.util
from pathlib import Path
import numpy as np
import pandas as pd
import h5py
import torch
import matplotlib.pyplot as plt
import matplotlib.tri as mtri
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D
from scipy.interpolate import RegularGridInterpolator
from scipy.spatial import cKDTree
from matplotlib.collections import LineCollection
from matplotlib.patches import FancyBboxPatch
import matplotlib as mpl

P = Path(os.environ.get('SPATIAL_TI_PROJECT', os.environ["STP_PROJECT"]))
D = Path(os.environ.get('QIAN_DATA_ROOT', os.environ["QIAN_PFC_DATA_ROOT"]))
REPO = Path(os.environ.get('FLOW_STP_REPO',os.environ["FLOW_STP_REPO"]))
if not (REPO/'src/flow_stp/stage1/stage1a.py').is_file():
    REPO = P
SRC = REPO/'src/flow_stp/stage1/stage1a.py'

RUN = P/'experiments/qian_pfc_spatial_ti_all9_synthetic_defaults_v1/CPU_FULL'
EN = P/'paper_figures/qian_ENETSP_spatial_transport_LATEST_3AGE_BARYCENTRIC_v1'
H2 = P/'paper_figures/qian_PFC_H2_H3_Fisher_transition_matrices_TEXT_v1/H2_7_TRANSITIONS_MOSCOT_STVCR_ZOETROPE_v1'
REC = H2/'STVCR_CORRECTED_NATIVE_EXPORT_VERIFIED_v1/stvcr_corrected_endpoint_export_receipt.json'
OUT = P/'paper_figures/qian_PFC_spatial_trajectory_MAIN_FIGURE_v3_STANDALONE'
OUT.mkdir(parents=True, exist_ok=True)

ATLAS = D/'merscope_integrated_855.h5ad'
BARY = EN/'ENETSP_GW15_to_GW20_GW22_BARYCENTRIC_EXPECTED_PATHS.csv.gz'
COHORT = EN/'GW15_ENETSP_source_cells_REDERIVED.csv.gz'
STV = {
    2: EN/'stVCR_NEW3AGE_GW15_to_GW20_6000_UPDATED.csv.gz',
    7: EN/'stVCR_NEW3AGE_GW15_to_GW22_6000_UPDATED.csv.gz'
}
STAGE = RUN/'tables/stage1a_all_cells.csv.gz'
CKPT = RUN/'model_final.pt'
SPECS = {
    1:('GW15','UMB1117','F1b',410492,0.),
    2:('GW20','FB080','F1',665204,5/7),
    7:('GW22','FB123','F2',439986,1.)
}

for f in [ATLAS,SRC,BARY,COHORT,REC,STAGE,CKPT,*STV.values(),*[D/f'{v[0].lower()}.h5ad' for v in SPECS.values()]]:
    if not f.is_file():
        raise FileNotFoundError(f'Missing saved input: {f}')

receipt = json.loads(REC.read_text())
qc = receipt.get('target_coordinate_validation', [])
if receipt.get('retrained') is not False or {q.get('age') for q in qc} != {'GW15','GW20','GW22'} or any(q['p99_fraction_of_diameter'] > .001 for q in qc):
    raise RuntimeError('No successful, frozen stVCR target-native coordinate conversion receipt')

def sha(f):
    h = hashlib.sha256()
    with open(f,'rb') as stream:
        for b in iter(lambda: stream.read(4 << 20), b''):
            h.update(b)
    return h.hexdigest()

if receipt.get('checkpoint_sha256') != sha(P/'experiments/qian_pfc_publication_4way_20261024/stvcr_slices_1_2_7_official/stvcr_model_full.pt'):
    raise RuntimeError('Corrected stVCR receipt refers to a different frozen checkpoint')

def dec(v):
    return v.decode() if isinstance(v,(bytes,np.bytes_)) else str(v)

def idx(h, sample, region):
    obs = h['obs']
    s = obs['sample']
    r = obs['region']
    ss = [dec(v) for v in s['categories'][:]]
    rr = [dec(v) for v in r['categories'][:]]
    return np.flatnonzero((s['codes'][:] == ss.index(sample)) & (r['codes'][:] == rr.index(region)))

def strings(obj, ix):
    if isinstance(obj, h5py.Group) and 'codes' in obj:
        cats = np.array([dec(v) for v in obj['categories'][:]], dtype=object)
        cc = np.asarray(obj['codes'][ix], int)
        return np.where(cc >= 0, cats[np.maximum(cc,0)], '').astype(str)
    return np.array([dec(v) for v in obj[ix]], dtype=str)

# ---------------- Raw tissue geometry / IDs / H3 reference domains ----------------
S = {}
for sid,(age,sample,region,n,t) in SPECS.items():
    with h5py.File(D/f'{age.lower()}.h5ad','r') as h:
        ii = idx(h,sample,region)
        if len(ii) != n:
            raise RuntimeError(f'{age} raw section changed: {len(ii)} != {n}')
        xy = np.asarray(h['obsm']['spatial'][ii,:2], np.float64)
        ids = strings(h['obs'][dec(h['obs'].attrs.get('_index','_index'))], ii)
        if not np.isfinite(xy).all():
            raise RuntimeError('Nonfinite tissue XY')
        S[sid] = dict(age=age,t=t,xy=xy,ids=ids)

with h5py.File(ATLAS,'r') as h:
    for sid,(_,sample,region,_,_) in SPECS.items():
        ii = idx(h,sample,region)
        xy = np.asarray(h['obsm']['spatial'][ii,:2], float)
        labels = strings(h['obs']['H3_annotation'], ii)
        mask = np.fromiter((str(v).startswith('EN-ET-SP-') for v in labels), bool, count=len(labels))
        S[sid]['h3xy'] = xy[mask]
        if sid == 1:
            tree = cKDTree(xy)
            S[1]['atlas_tree'] = tree
            S[1]['atlas_h3'] = mask.copy()
            query = xy[np.linspace(0, len(xy)-1, min(2100, len(xy)), dtype=int)]
            spacing = float(np.median(tree.query(query, k=6)[0][:,5]))
            dd, nn = tree.query(S[1]['xy'], k=1, workers=-1)
            source_rows = np.flatnonzero((dd <= .55*spacing) & mask[nn])
        else:
            tree = cKDTree(xy)
            query = xy[np.linspace(0, len(xy)-1, min(2100, len(xy)), dtype=int)]
            S[sid]['spacing'] = float(np.median(tree.query(query, k=6)[0][:,5]))
S[1]['spacing'] = spacing

cohort = pd.read_csv(COHORT, dtype={'cell_id':str})
actual_ids = np.char.add('s01_GW15:', S[1]['ids'][source_rows])
if not np.array_equal(cohort.stage_row.to_numpy(np.int64), source_rows) or not np.array_equal(cohort.cell_id.to_numpy(str), actual_ids):
    raise RuntimeError('Reconstructed H3 source cohort differs from original frozen cohort')
print('PASS original EN-ET-SP H3 source cohort:', len(source_rows), 'cells')

# ---------------- Re-read saved nine-section anatomical coordinates ----------------
counts = np.zeros(9, dtype=np.int64)
zblocks = {sid:[] for sid in SPECS}
allparts = []
header = set(pd.read_csv(STAGE, nrows=0).columns)
xycol = next((p for p in [('spatial_x','spatial_y'),('x','y'),('raw_x','raw_y'),('physical_x','physical_y')] if set(p) <= header), None)
idcol = next((c for c in ('original_cell_id','raw_cell_id','obs_name','cell_id') if c in header), None)
columns = ['slice_id','z_stage1a'] + list(xycol or []) + ([idcol] if idcol else [])

for ch in pd.read_csv(STAGE, usecols=columns, chunksize=300000):
    ids = ch.slice_id.to_numpy(int)
    counts += np.bincount(ids, minlength=9)
    allparts.append(ch.z_stage1a.to_numpy(float))
    for sid in SPECS:
        part = ch.loc[ids == sid]
        if len(part):
            zblocks[sid].append(part)

if counts.tolist() != [548567,410492,665204,596754,322648,553738,508417,439986,333113]:
    raise RuntimeError('Frozen nine-section Stage-1a table cell counts changed')

for sid in SPECS:
    df = pd.concat(zblocks[sid], ignore_index=True)
    S[sid]['z'] = df.z_stage1a.to_numpy(float)
    if xycol:
        error = np.linalg.norm(df[list(xycol)].to_numpy(float) - S[sid]['xy'], axis=1)
        if np.quantile(error,.99) > 1e-3:
            raise RuntimeError('Stage1 z vs original raw tissue row order mismatch')
    elif idcol:
        obs = df[idcol].astype(str).to_numpy()
        original = S[sid]['ids']
        choices = [
            original,
            np.char.add(f's{sid}:',original),
            np.char.add(f'{sid}|',original),
            np.char.add(f's{sid:02d}_{S[sid]["age"]}:',original)
        ]
        if not any(np.array_equal(obs,x) for x in choices):
            print(f'NOTICE: sid {sid} Stage-1a IDs generated; assuming original per-section row order')
    else:
        print(f'NOTICE: sid {sid} Stage-1a coordinates have no ID/XY join; assuming original per-section row order')

allz = np.concatenate(allparts)
levels = np.quantile(S[1]['z'][source_rows], np.linspace(.18,.82,5))
if len(np.unique(levels)) != 5:
    raise RuntimeError('Collapsed EN-ET-SP anatomical contour levels')

# ---------------- Frozen Fisher velocity and RK4 integration ----------------
ck = torch.load(CKPT, map_location='cpu', weights_only=False)
if ck.get('method_version') != 'stage1a_shared_logE_global_gauge_v1':
    raise RuntimeError('Wrong Zoetrope model version')

spec = importlib.util.spec_from_file_location('pfc_en_standalone_stage1a', SRC)
mod = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = mod
spec.loader.exec_module(mod)

conf = {k:v for k,v in ck['model_config'].items() if k in mod.FlowSTPConfig.__dataclass_fields__}
model = mod.FlowSTPModel(mod.FlowSTPConfig(**conf))
model.load_state_dict(ck['model_state_dict'], strict=True)
model.eval().cpu()
torch.set_num_threads(min(6, os.cpu_count() or 2))

tg = np.linspace(0,1,121)
zg = np.linspace(float(np.quantile(allz,.0001)-.27), float(np.quantile(allz,.9999)+.27), 241)
T, Z = np.meshgrid(tg, zg, indexing='ij')
vfield = np.empty(T.size, float)
for a in range(0, T.size, 768):
    b = min(a+768, T.size)
    z = torch.as_tensor(Z.ravel()[a:b], dtype=torch.float32)
    t = torch.as_tensor(T.ravel()[a:b], dtype=torch.float32)
    with torch.enable_grad():
        _, dz = torch.func.jvp(model.fisher_flat, (z,t), (torch.ones_like(z), torch.zeros_like(t)))
        _, dt = torch.func.jvp(model.fisher_flat, (z,t), (torch.zeros_like(z), torch.ones_like(t)))
    vfield[a:b] = (-((dz*dt).sum(1)/(dz.square().sum(1)+1e-8))).detach().numpy()

if not np.isfinite(vfield).all():
    raise RuntimeError('Nonfinite Fisher velocity field')

V = RegularGridInterpolator((tg,zg), vfield.reshape(T.shape), bounds_error=False, fill_value=np.nan)

def advect(vals, target_sid):
    if target_sid == 1:
        return np.asarray(vals,float)
    y = np.asarray(vals,float).copy()
    a = 0.
    b = S[target_sid]['t']
    n = max(64, int(round(320*(b-a))))
    h = (b-a)/n
    def f(t,x):
        return V(np.column_stack((np.full(len(x), t), x)))
    for j in range(n):
        t = a + j*h
        k1 = f(t,y)
        k2 = f(t+h/2, y+h*k1/2)
        k3 = f(t+h/2, y+h*k2/2)
        k4 = f(t+h, y+h*k3)
        y += h*(k1 + 2*k2 + 2*k3 + k4)/6
    if not np.isfinite(y).all():
        raise RuntimeError('Fisher contour outside training field support')
    return y

levels_at = {sid: advect(levels, sid) for sid in SPECS}
print('PASS reconstructed original five Fisher contour levels')

# ---------------- Original adaptive-bin physical contour mesh ----------------
def mesh(sid):
    x = S[sid]['xy']
    z = S[sid]['z']
    width = 3.25*S[sid]['spacing']
    maxcells = 23500
    for _ in range(9):
        bins = np.floor((x-x.min(0))/width).astype(np.int32)
        unique,inv = np.unique(bins, axis=0, return_inverse=True)
        if len(unique) <= maxcells:
            break
        width *= max(1.14, np.sqrt(len(unique)/maxcells)*1.04)
    n = len(unique)
    count = np.bincount(inv, minlength=n)
    xx = np.bincount(inv, weights=x[:,0], minlength=n)/count
    yy = np.bincount(inv, weights=x[:,1], minlength=n)/count
    zz = np.bincount(inv, weights=z, minlength=n)/count
    tri = mtri.Triangulation(xx,yy)
    face = tri.triangles
    length = np.maximum.reduce([
        np.hypot(xx[face[:,a]]-xx[face[:,b]], yy[face[:,a]]-yy[face[:,b]])
        for a,b in ((0,1),(1,2),(0,2))
    ])
    tri.set_mask(length > 4.3*width)
    return tri,zz

MESH = {sid: mesh(sid) for sid in SPECS}

def contour_paths(tri, zvals, levels):
    vals = np.asarray(levels,float)
    vals = np.unique(np.sort(vals[(vals > np.min(zvals)) & (vals < np.max(zvals))]))
    if not len(vals):
        raise RuntimeError('No valid contour levels within saved z triangulation')
    temp,ax = plt.subplots(figsize=(1,1))
    try:
        cs = ax.tricontour(tri, zvals, levels=vals, colors='k')
        paths = [np.asarray(p,float) for family in cs.allsegs for p in family if len(p)>=2 and np.isfinite(p).all()]
    finally:
        plt.close(temp)
    if not paths:
        raise RuntimeError('Triangulation produced no finite contour paths')
    return paths

def segments_at_resolution(paths, step):
    if not np.isfinite(step) or step <= 0:
        raise ValueError('Bad contour sampling step')
    pieces = []
    for path in paths:
        diff = np.diff(path, axis=0)
        k = np.maximum(1, np.ceil(np.linalg.norm(diff,axis=1)/step).astype(np.int32))
        start = np.repeat(path[:-1], k, axis=0)
        dr = np.repeat(diff, k, axis=0)
        accum = np.cumsum(k)
        j = np.arange(int(accum[-1])) - np.repeat(accum-k, k)
        frac = j / np.repeat(k,k)
        delta = dr / np.repeat(k,k)[:,None]
        p0 = start + frac[:,None]*dr
        pieces.append(np.stack([p0, p0+delta], axis=1))
    out = np.concatenate(pieces)
    if not len(out):
        raise RuntimeError('Empty source contour line segments')
    return out

RED = '#D62728'
BLACK = '#1F2428'
CYAN = '#00A6BD'
GRAY = '#969EA6'
INK = '#26343D'
UNKNOWN = '#A1A6AC'

BAND_RADIUS_FACTOR = 2.5
PROXIMITY_RADIUS_FACTOR = 4.0
PROXIMITY_MIN_KNOWN_FOR_NEGATIVE = 4
PROXIMITY_RADIUS_SWEEP = (1.0, 2.0, 3.0, 4.0, 6.0)
TARGET_VOTE_K = 21
TARGET_VOTE_RADIUS_FACTOR = 3.0
TARGET_VOTE_MIN_KNOWN = 7
TARGET_VOTE_THRESHOLD = 0.50
RAW_COVERAGE_FACTOR = 1.50
SEED_DRAW = 20261103
DPI = 450

mpl.rcParams.update({
    'font.family':'DejaVu Sans','font.size':8.6,
    'axes.titlesize':9.2,'axes.titleweight':'bold',
    'axes.labelsize':8.8,'xtick.labelsize':8,'ytick.labelsize':8,
    'pdf.fonttype':42,'ps.fonttype':42,'svg.fonttype':'none',
    'savefig.facecolor':'white','figure.facecolor':'white',
})

def contour_touched(source_xy, paths, radius):
    segs = segments_at_resolution(paths, step=radius/3.)
    mid = segs.mean(axis=1)
    distances,_ = cKDTree(mid).query(source_xy, k=1, workers=-1)
    selected = np.isfinite(distances) & (distances <= radius)
    return selected,distances

def local_5nn(xy, k=6):
    xy = np.asarray(xy,float)
    ix = np.linspace(0, len(xy)-1, min(3000, len(xy)), dtype=int)
    return float(np.median(cKDTree(xy).query(xy[ix], k=k, workers=-1)[0][:,5]))

def make_truth(atlas, target_sid, spec, label_key, acceptable, raw_tissue_xy):
    age,sample,region = spec[:3]
    with h5py.File(atlas,'r') as h:
        ii = idx(h,sample,region)
        xy = np.asarray(h['obsm']['spatial'][ii,:2], np.float64)
        labels = strings(h['obs'][label_key], ii)
    ok = np.array([bool(acceptable(v)) for v in labels], dtype=bool)
    missing = {'','nan','none','na','n/a','unknown','unassigned','unannotated','unlabeled'}
    known = np.array([str(v).strip().lower() not in missing for v in labels], dtype=bool)
    assert np.all(~ok | known), 'Target H3-positive cells include missing annotation values'
    if not np.any(ok):
        raise RuntimeError(f'No expected H3 domain in target sid={target_sid}')
    spacing = local_5nn(xy)
    raw_tissue_xy = np.asarray(raw_tissue_xy, dtype=np.float64)
    raw_spacing = local_5nn(raw_tissue_xy)
    return dict(
        xy=xy, labels=labels, good=ok, label_known=known, tree=cKDTree(xy),
        good_tree=cKDTree(xy[ok]),
        proximity_radius=PROXIMITY_RADIUS_FACTOR*spacing,
        min_known_negative=PROXIMITY_MIN_KNOWN_FOR_NEGATIVE,
        vote_radius=TARGET_VOTE_RADIUS_FACTOR*spacing,
        vote_k=TARGET_VOTE_K, min_known=TARGET_VOTE_MIN_KNOWN,
        vote_threshold=TARGET_VOTE_THRESHOLD, spacing=spacing,
        raw_tree=cKDTree(raw_tissue_xy), raw_spacing=raw_spacing,
        raw_max_dist=RAW_COVERAGE_FACTOR*raw_spacing,
        n_good=int(ok.sum()), sid=target_sid
    )

def classify_region_support(xy, truth):
    xy = np.asarray(xy,float)
    d,ix = truth['tree'].query(xy, k=truth['vote_k'],
                               distance_upper_bound=truth['vote_radius'],
                               workers=-1)
    d = np.asarray(d).reshape(len(xy),-1)
    ix = np.asarray(ix).reshape(len(xy),-1)
    valid = np.isfinite(d) & (ix < truth['tree'].n)
    safe_ix = np.where(valid, ix, 0)
    known_votes = valid & truth['label_known'][safe_ix]
    correct_votes = known_votes & truth['good'][safe_ix]
    n_known = known_votes.sum(axis=1)
    n_correct = correct_votes.sum(axis=1)
    n_nearby = valid.sum(axis=1)
    vote_fraction = np.divide(n_correct, n_known, out=np.full(len(xy),np.nan), where=n_known>0)
    atlas_dist = d[:,0]
    raw_dist,_ = truth['raw_tree'].query(xy, k=1, workers=-1)
    on_raw = np.isfinite(raw_dist) & (raw_dist <= truth['raw_max_dist'])
    nearest_expected_dist,_ = truth['good_tree'].query(xy, k=1, workers=-1)
    near_expected = (nearest_expected_dist <= truth['proximity_radius'])
    strict_majority = (on_raw & (n_known >= truth['min_known']) & (vote_fraction > truth['vote_threshold']))
    correct = on_raw & near_expected
    annotated = on_raw & ((n_known >= truth['min_known_negative']) | near_expected)
    wrong = annotated & ~correct
    unresolved = on_raw & ~annotated
    outside = ~on_raw

    labels = np.full(len(xy),'INSUFFICIENT_H3_COVERAGE',dtype=object)
    labels[correct] = 'NEAR_EN_ET_SP_TERRITORY'
    labels[wrong] = 'NO_EXPECTED_H3_WITHIN_PROXIMITY_RADIUS'
    labels[outside] = 'OUTSIDE_RAW_TISSUE_COVERAGE'

    status = np.full(len(xy),'unresolved',dtype=object)
    status[correct] = 'near_expected_H3'
    status[wrong] = 'not_near_expected_H3'
    status[outside] = 'outside_raw_coverage'

    return dict(
        correct=correct, wrong=wrong, unresolved=unresolved,
        outside=outside, on_raw=on_raw, annotated=annotated,
        atlas_dist=atlas_dist, raw_dist=raw_dist,
        n_neighbors=n_nearby, n_known=n_known, n_correct_votes=n_correct,
        vote_fraction=vote_fraction, strict_majority=strict_majority,
        nearest_expected_dist=nearest_expected_dist,
        nearest_expected_dist_in_5nn=nearest_expected_dist/truth['spacing'],
        labels=np.asarray(labels,dtype=str), status=np.asarray(status,dtype=str)
    )

def fraction(mask, weights):
    weights = np.asarray(weights, dtype=float)
    return float(weights[np.asarray(mask,bool)].sum()/weights.sum()) if weights.sum() > 0 else float('nan')

def region_summary(c, weights):
    covered = c['annotated']
    w = np.asarray(weights,float)
    return dict(
        segment_support_fraction=fraction(c['correct'],w),
        correct_given_annotated_fraction=(fraction(c['correct'],w)/fraction(covered,w) if fraction(covered,w)>0 else float('nan')),
        annotated_fraction=fraction(covered,w),
        incorrect_label_fraction=fraction(c['wrong'],w),
        atlas_unresolved_fraction=fraction(c['unresolved'],w),
        off_tissue_fraction=fraction(c['outside'],w),
    )

def colored_contour_segments(paths, truth):
    segments = segments_at_resolution(paths, truth['spacing']*.85)
    classification = classify_region_support(segments.mean(axis=1), truth)
    lengths = np.linalg.norm(segments[:,1]-segments[:,0], axis=1)
    colors = np.full(len(segments), UNKNOWN, dtype=object)
    colors[classification['correct']] = RED
    colors[classification['wrong']] = BLACK
    summary = region_summary(classification, lengths)
    summary.update(
        n_segments=len(segments),
        n_evaluable=int(classification['annotated'].sum()),
        segments=segments,
        segment_lengths=lengths,
        classification=classification
    )
    return segments,colors,summary

def source_draw(ax, xy, ref_xy, selected_xy, paths, bounds, src_label):
    lo,hi = bounds
    ax.scatter(xy[:,0], xy[:,1], s=.24, c=GRAY, alpha=.23, linewidths=0, rasterized=True, zorder=1)
    if len(ref_xy):
        ax.scatter(ref_xy[:,0], ref_xy[:,1], s=.48, c=CYAN, alpha=.56, linewidths=0, rasterized=True, zorder=2)
    for path in paths:
        ax.plot(path[:,0], path[:,1], color=BLACK, lw=.58, alpha=.65, zorder=3)
    ax.scatter(selected_xy[:,0], selected_xy[:,1], s=3.6, c=RED, alpha=.92, linewidths=0, rasterized=True, zorder=5)
    ax.set_xlim(lo[0],hi[0])
    ax.set_ylim(hi[1],lo[1])
    ax.set_aspect('equal')
    ax.axis('off')

def target_draw(ax, background, reference, bounds, truth, point_xy=None, ti_paths=None):
    lo,hi = bounds
    ax.scatter(background[:,0], background[:,1], s=.25, c=GRAY, alpha=.18,
               linewidths=0, rasterized=True, zorder=1)
    if len(reference):
        ax.scatter(reference[:,0], reference[:,1], s=.52, c=CYAN,
                   alpha=.58, linewidths=0, rasterized=True, zorder=2)
    info = {}
    if ti_paths is not None:
        seg,col,info = colored_contour_segments(ti_paths,truth)
        ax.add_collection(LineCollection(seg, colors=col, linewidths=1.13,
                                         alpha=.96, zorder=6, rasterized=False))
    elif point_xy is not None:
        c = classify_region_support(point_xy,truth)
        for mask,color,s,alpha in ((c['unresolved']|c['outside'],UNKNOWN,3.6,.69),
                                   (c['wrong'],BLACK,3.9,.90),
                                   (c['correct'],RED,4.2,.91)):
            if mask.any():
                ax.scatter(point_xy[mask,0], point_xy[mask,1], s=s,
                           c=color, alpha=alpha, linewidths=0,
                           zorder=6, rasterized=True)
        summary = region_summary(c, np.ones(len(point_xy),float))
        info = dict(
            n=len(point_xy), n_correct=int(c['correct'].sum()),
            n_in_tissue=int(c['on_raw'].sum()),
            fraction_correct=float(c['correct'].mean()),
            distances=c['atlas_dist'], predicted_labels=c['labels'],
            correct=c['correct'], inside=c['on_raw'],
            status=c['status'], atlas_dist=c['atlas_dist'], raw_dist=c['raw_dist'],
            n_neighbors=c['n_neighbors'], n_known=c['n_known'],
            n_correct_votes=c['n_correct_votes'], vote_fraction=c['vote_fraction'],
            nearest_expected_dist=c['nearest_expected_dist'],
            nearest_expected_dist_in_5nn=c['nearest_expected_dist_in_5nn'],
            strict_majority=c['strict_majority'], classification=c,
            **summary
        )
    else:
        raise ValueError('Provide precisely one prediction type')
    ax.set_xlim(lo[0],hi[0])
    ax.set_ylim(hi[1],lo[1])
    ax.set_aspect('equal')
    ax.axis('off')
    return info

def fixed_background(xy, reference, sid, max_bg=24000, max_ref=18000):
    rng = np.random.default_rng(SEED_DRAW+sid)
    bi = np.sort(rng.choice(len(xy), size=min(max_bg,len(xy)), replace=False))
    ri = np.sort(rng.choice(len(reference), size=min(max_ref,len(reference)), replace=False))
    pad = np.maximum(np.ptp(xy,axis=0)*.028, 1e-5)
    return xy[bi],reference[ri],(xy.min(0)-pad,xy.max(0)+pad)

def safe_export(fig,out,base):
    for suffix in ('png','pdf','svg'):
        p = out/(base+'.'+suffix)
        fig.savefig(p, dpi=DPI if suffix=='png' else None,
                    facecolor='white', bbox_inches=None, pad_inches=0)
        assert p.is_file() and p.stat().st_size > 500, f'Figure failed to save: {p}'
        print('SAVED', p, 'size', p.stat().st_size, 'bytes')

# ---------------- H3 EN-ET-SP | identical 6,000-source universe ----------------
OUT = P/'paper_figures/qian_PFC_spatial_trajectory_H3_v7_BOTTOM_CROPPED_4X5NN'
OUT.mkdir(parents=True, exist_ok=True)

TRUTH = {}
for sid in (2,7):
    TRUTH[sid] = make_truth(ATLAS, sid, SPECS[sid], 'H3_annotation',
                            lambda s: str(s).startswith('EN-ET-SP-'),
                            S[sid]['xy'])
    print(S[sid]['age'], 'H3 target cells', TRUTH[sid]['n_good'],
          'proximity radius (4x atlas 5NN)', round(TRUTH[sid]['proximity_radius'],3),
          'min known labels to mark negative', TRUTH[sid]['min_known_negative'],
          'raw tissue radius', round(TRUTH[sid]['raw_max_dist'],3))
    _rr = np.random.default_rng(20261103+sid)
    _probe = S[sid]['xy'][_rr.choice(len(S[sid]['xy']), size=min(8000,len(S[sid]['xy'])), replace=False)]
    _d,_ = TRUTH[sid]['tree'].query(_probe, k=1, workers=-1)
    _coverage = float(np.mean(_d <= TRUTH[sid]['proximity_radius']))
    print(S[sid]['age'], f'raw-to-atlas nearest distance within voting radius: {_coverage:.1%}',
          'raw-to-atlas NN p50/p95:', np.quantile(_d,[.5,.95]).round(3))

mos = pd.read_csv(BARY, dtype={'gw15_cell_id':str}).set_index('gw15_cell_id', verify_integrity=True)
sv = {sid: pd.read_csv(STV[sid], dtype={'source_cell_id':str}) for sid in (2,7)}
if not sv[2].source_cell_id.equals(sv[7].source_cell_id) or len(sv[2]) != 6000:
    raise RuntimeError('Corrected stVCR GW20/22 source cohort changed')
ids = sv[2].source_cell_id.to_numpy(str)
if not np.isin(ids, cohort.cell_id).all():
    raise RuntimeError('Corrected stVCR sources absent from H3 cohort')
mos = mos.reindex(ids)
if mos[['gw20_x','gw20_y','gw22_x','gw22_y']].isna().any().any():
    raise RuntimeError('One or more matched moscot barycenters absent')
for sid in (2,7):
    xy = sv[sid][['x_pred_native','y_pred_native']].to_numpy(float)
    if not np.isfinite(xy).all():
        raise RuntimeError('Nonfinite corrected stVCR positions')
print('PASS moscot/stVCR exact matched 6000 sources for both target ages')

sv_ids = sv[2].source_cell_id.astype(str).to_numpy()
assert len(sv_ids) == 6000 and len(set(sv_ids)) == len(sv_ids)
assert np.array_equal(sv[7].source_cell_id.astype(str).to_numpy(), sv_ids)
assert len(set(cohort.cell_id.astype(str))) == len(cohort)
meta = cohort.set_index('cell_id').reindex(sv_ids)
if meta.stage_row.isna().any():
    raise RuntimeError('Missing source identities in original H3 cohort')

source_rows = meta.stage_row.to_numpy(np.int64)
assert len(set(source_rows)) == len(source_rows)
assert np.isin(sv_ids, mos.index.astype(str)).all()
source_xy = S[1]['xy'][source_rows]
source_z = S[1]['z'][source_rows]

assert len(S[1]['atlas_h3']) == S[1]['atlas_tree'].n
_src_dist,_src_nearest = S[1]['atlas_tree'].query(source_xy, k=1, workers=-1)
assert np.all((_src_dist <= .55*S[1]['spacing']) & S[1]['atlas_h3'][_src_nearest]), (
    'Frozen H3 source cohort no longer maps to EN-ET-SP labels'
)

source_paths = contour_paths(*MESH[1], levels_at[1])
source_radius = BAND_RADIUS_FACTOR*S[1]['spacing']
touched,distances = contour_touched(source_xy, source_paths, source_radius)
if touched.sum() < 15:
    raise RuntimeError(f'Only {touched.sum()} shared cells are physically within {source_radius:.4g} of source contours.')

print('H3 shared contour-touched cohort:', int(touched.sum()), '/ 6000; physical band', source_radius)

selected_ids = sv_ids[touched]
selected_xy = source_xy[touched]
selected_rows = source_rows[touched]
selected_z = source_z[touched]

selected = pd.DataFrame({
    'source_cell_id':selected_ids,
    'stage_row':selected_rows,
    'source_x':selected_xy[:,0],
    'source_y':selected_xy[:,1],
    'source_z':selected_z,
    'distance_to_contour':distances[touched]
})
for sid in (2,7):
    age = S[sid]['age']
    selected[f'{age}_zoetrope_z'] = advect(selected_z, sid)
selected.to_csv(OUT/'MATCHED_GW15_H3_CONTOUR_TOUCHING_SOURCE_CELLS.csv', index=False)

PLOTS = {}
for sid in SPECS:
    if sid == 1:
        x = S[sid]['xy']
        r = S[sid]['h3xy']
    else:
        x = S[sid]['xy']
        r = TRUTH[sid]['xy'][TRUTH[sid]['good']]
    PLOTS[sid] = fixed_background(x,r,sid)

REFERENCE_BACKGROUND = {}
for sid in (2,7):
    bg = PLOTS[sid][0]
    c_bg = classify_region_support(bg, TRUTH[sid])
    REFERENCE_BACKGROUND[sid] = c_bg
    print(S[sid]['age'], 'near-region reference on random raw tissue:',
          f'{100*c_bg["correct"].mean():.1f}%', 'of', len(bg), 'raw cells', flush=True)

# ================================================================
# FIGURE | Two aligned rows: GW15 opposite GW20, legend opposite GW22
# Internal drawing canvas: 5.50 x 5.40 in; bottom-only cropped output.
# ================================================================
FIG_W, FIG_H = 5.50, 5.40
METHODS = ('Zoetrope', 'moscot', 'stVCR')

TARGET_COL_LEFTS = (1.77, 3.02, 4.27)
TARGET_SLOT_W = 1.20
TOP_TISSUE = 5.06
ROW_GAP_IN = 0.30
LOWER_MIN_BOTTOM = 0.40

def native_ratio(bounds):
    lo,hi = bounds
    dx = float(hi[0]-lo[0])
    dy = float(hi[1]-lo[1])
    if dx <= 0 or dy <= 0 or not np.isfinite([dx,dy]).all():
        raise ValueError(f'Invalid physical tissue bounds: {bounds}')
    return dy/dx

ratio20 = native_ratio(PLOTS[2][2])
ratio22 = native_ratio(PLOTS[7][2])
max_available = TOP_TISSUE - LOWER_MIN_BOTTOM - ROW_GAP_IN
actual_target_w = min(TARGET_SLOT_W, max_available/(ratio20 + ratio22))
if actual_target_w < 0.60:
    raise RuntimeError('Target aspect ratios need manual review')
TARGET_WIDTHS = {2:actual_target_w, 7:actual_target_w}
TARGET_HEIGHTS = {2:actual_target_w*ratio20, 7:actual_target_w*ratio22}
TARGET_ROW_TOPS = {2:TOP_TISSUE, 7:TOP_TISSUE - TARGET_HEIGHTS[2] - ROW_GAP_IN}
TARGET_ROW_BOTTOMS = {sid:TARGET_ROW_TOPS[sid]-TARGET_HEIGHTS[sid] for sid in (2,7)}
TARGET_ROWS_IN = ((2,'GW20',TARGET_ROW_BOTTOMS[2]), (7,'GW22',TARGET_ROW_BOTTOMS[7]))

mpl.rcParams.update({
    'font.family':'DejaVu Sans','font.size':9.2,
    'axes.titlesize':10.1,'axes.titleweight':'bold',
    'axes.labelsize':9.5,'xtick.labelsize':8.7,'ytick.labelsize':8.7,
    'axes.linewidth':.7,'text.color':INK,'axes.labelcolor':INK,
    'pdf.fonttype':42,'ps.fonttype':42,'svg.fonttype':'none',
    'savefig.facecolor':'white','figure.facecolor':'white',
})

fig = plt.figure(figsize=(FIG_W,FIG_H), facecolor='white')

# ---------------- GW15 source ----------------
bg,ref,bounds = PLOTS[1]
src_ratio = native_ratio(bounds)
src_w = min(1.49, 1.02/src_ratio)
src_h = src_w*src_ratio
SRC_CENTER_X = 0.86
upper_center = (TARGET_ROW_TOPS[2]+TARGET_ROW_BOTTOMS[2])/2
src_left = SRC_CENTER_X-src_w/2
src_bottom = upper_center-src_h/2

ax0 = fig.add_axes([src_left/FIG_W, src_bottom/FIG_H, src_w/FIG_W, src_h/FIG_H])
source_draw(ax0, bg, ref, selected_xy, source_paths, bounds, 'EN-ET-SP')
fig.text(SRC_CENTER_X/FIG_W, (src_bottom+src_h+.155)/FIG_H,
         'GW15 · EN-ET-SP', fontsize=10.1, fontweight='bold',
         ha='center', va='bottom', color=INK)
fig.text(SRC_CENTER_X/FIG_W, (src_bottom-.10)/FIG_H,
         f'{len(selected_ids):,} source cells', fontsize=8.5,
         ha='center', va='top', color=INK)

# ---------------- Legend box: shifted left and slightly narrower ----------------
leg_w, leg_h = 1.38, 1.14
leg_center_x = 0.78   # shifted left so it clears the GW22 vertical label
leg_x = leg_center_x - leg_w/2
lower_center = (TARGET_ROW_TOPS[7]+TARGET_ROW_BOTTOMS[7])/2
leg_y = lower_center - leg_h/2

ax_leg = fig.add_axes([leg_x/FIG_W, leg_y/FIG_H, leg_w/FIG_W, leg_h/FIG_H])
ax_leg.axis('off')
ax_leg.add_patch(FancyBboxPatch((.015,.016), .97, .968,
    boxstyle='round,pad=0.004,rounding_size=0.025',
    transform=ax_leg.transAxes, facecolor='white',
    edgecolor='#C8CFD5', linewidth=.70, zorder=0))

legend_items = [
    ('Labeled EN-ET-SP', CYAN),
    ('Correct type', RED),
    ('Incorrect type', BLACK),
    ('Unassigned', UNKNOWN),
]
for i,(lab,color) in enumerate(legend_items):
    y = .81 - i*.205
    ax_leg.scatter([.11],[y], s=28, color=color, linewidths=0,
                   transform=ax_leg.transAxes, zorder=2)
    ax_leg.text(.205, y, lab, ha='left', va='center', color=INK,
                fontsize=8.15, transform=ax_leg.transAxes, zorder=2)

# ---------------- Method headings ----------------
for col,method in enumerate(METHODS):
    cx = TARGET_COL_LEFTS[col] + TARGET_SLOT_W/2
    fig.text(cx/FIG_W, 5.20/FIG_H, method,
             fontsize=10.1, fontweight='bold',
             ha='center', va='bottom', color=INK)

rows = []
RESULT_CLASSIFICATIONS = {}
for sid,rowname,_ in TARGET_ROWS_IN:
    truth = TRUTH[sid]
    bg,ref,bounds = PLOTS[sid]
    actual_w = TARGET_WIDTHS[sid]
    actual_h = TARGET_HEIGHTS[sid]
    real_bottom = TARGET_ROW_BOTTOMS[sid]

    fig.text(1.665/FIG_W, (real_bottom+actual_h/2)/FIG_H, rowname,
             ha='center', va='center', rotation=90,
             fontsize=9.3, fontweight='bold', color=INK)

    ti_levels = np.asarray(levels_at[sid],float)
    ti_paths = contour_paths(*MESH[sid], ti_levels)
    original_mos = mos.reindex(selected_ids)
    age = S[sid]['age'].lower()

    for col,method in enumerate(METHODS):
        x = TARGET_COL_LEFTS[col] + (TARGET_SLOT_W-actual_w)/2
        ax = fig.add_axes([x/FIG_W, real_bottom/FIG_H, actual_w/FIG_W, actual_h/FIG_H])

        if method == 'Zoetrope':
            stat = target_draw(ax, bg, ref, bounds, truth, ti_paths=ti_paths)
            _c = stat['classification']
            _seg = stat['segments']
            _mids = _seg.mean(axis=1)
            pd.DataFrame(dict(
                x=_mids[:,0], y=_mids[:,1],
                segment_length=stat['segment_lengths'], status=_c['status'],
                near_EN_ET_SP_territory=_c['correct'],
                distance_to_raw_tissue=_c['raw_dist'],
                distance_to_nearest_atlas_cell=_c['atlas_dist'],
                number_nearby_atlas_cells=_c['n_neighbors'],
                number_known_H3_neighbors=_c['n_known'],
                number_matching_H3_neighbors=_c['n_correct_votes'],
                local_target_H3_fraction=_c['vote_fraction'],
                strict_majority_21NN=_c['strict_majority'],
                nearest_H3_positive_distance=_c['nearest_expected_dist'],
                nearest_H3_positive_distance_in_5NN=_c['nearest_expected_dist_in_5nn']
            )).to_csv(OUT/f'{rowname}_ZOETROPE_CONTOUR_SEGMENT_AUDIT.csv.gz', index=False)
            percent_correct = 100*stat['segment_support_fraction']
        else:
            if method == 'moscot':
                xy = original_mos[[age+'_x', age+'_y']].to_numpy(float)
            else:
                xy = sv[sid].loc[touched, ['x_pred_native','y_pred_native']].to_numpy(float)
            stat = target_draw(ax, bg, ref, bounds, truth, point_xy=xy)
            percent_correct = 100*stat['fraction_correct']

            for column,values in [
                (f'{rowname}_{method}_x',xy[:,0]),
                (f'{rowname}_{method}_y',xy[:,1]),
                (f'{rowname}_{method}_near_expected_H3',stat['correct']),
                (f'{rowname}_{method}_in_tissue',stat['inside']),
                (f'{rowname}_{method}_assigned_H3',stat['predicted_labels']),
                (f'{rowname}_{method}_evaluation_status',stat['status']),
                (f'{rowname}_{method}_dist_to_raw',stat['raw_dist']),
                (f'{rowname}_{method}_dist_to_atlas',stat['atlas_dist']),
                (f'{rowname}_{method}_n_nearby',stat['n_neighbors']),
                (f'{rowname}_{method}_n_known',stat['n_known']),
                (f'{rowname}_{method}_n_matching',stat['n_correct_votes']),
                (f'{rowname}_{method}_local_H3_fraction',stat['vote_fraction']),
                (f'{rowname}_{method}_strict_majority',stat['strict_majority']),
                (f'{rowname}_{method}_nearest_positive_H3_distance',stat['nearest_expected_dist']),
                (f'{rowname}_{method}_nearest_positive_distance_in_5NN',stat['nearest_expected_dist_in_5nn'])
            ]:
                selected[column] = values

        RESULT_CLASSIFICATIONS[(sid,method)] = {
            'classification': stat['classification'],
            'weights': (stat['segment_lengths'] if method=='Zoetrope' else np.ones(len(selected_ids),float))
        }

        rows.append(dict(
            age=rowname, method=method, n_source=len(selected_ids),
            n_evaluated=int(stat.get('n',stat.get('n_segments'))),
            n_near_expected_H3=stat.get('n_correct',np.nan),
            near_fraction=stat.get('fraction_correct',np.nan),
            contour_proximity_fraction=stat.get('segment_support_fraction',np.nan),
            annotation_coverage_fraction=stat['annotated_fraction'],
            confirmed_other_label_fraction=stat['incorrect_label_fraction'],
            unknown_atlas_fraction=stat['atlas_unresolved_fraction'],
            off_raw_tissue_fraction=stat['off_tissue_fraction'],
            near_given_evaluable_fraction=stat['correct_given_annotated_fraction'],
            raw_background_near_fraction=float(REFERENCE_BACKGROUND[sid]['correct'].mean()),
            strict_majority_fraction=(fraction(
                stat['classification']['strict_majority'],
                stat['segment_lengths'] if method=='Zoetrope' else np.ones(len(stat['classification']['strict_majority']),float)
            )),
            physical_band_radius=source_radius
        ))

        fig.text((TARGET_COL_LEFTS[col]+TARGET_SLOT_W/2)/FIG_W,
                 (real_bottom-.075)/FIG_H,
                 f'{percent_correct:.0f}% correct',
                 ha='center', va='top',
                 fontsize=8.8, fontweight='medium', color=INK)

# -------------------------------------------------------------
# TRIM BOTTOM CANVAS ONLY: same x/y locations, same dot sizes,
# same aspect ratio of each tissue, and same editable vectors.
# Do NOT use tight bbox, which would crop the other edges too.
# -------------------------------------------------------------
from matplotlib.transforms import Bbox

fig.canvas.draw()
_renderer_crop = fig.canvas.get_renderer()
_to_inch_crop = fig.dpi_scale_trans.inverted()
_caption_objs = [t for t in fig.texts if t.get_text().strip().endswith('% correct')]
if len(_caption_objs) != 6:
    raise RuntimeError('Expected 6 GW20/GW22 percentage captions for this H3 figure.')
_caption_bottom_inches = min(
    t.get_window_extent(renderer=_renderer_crop).transformed(_to_inch_crop).y0
    for t in _caption_objs
)
# GW22 is the lower row. The legend and tissue axes must remain inside the crop.
_lowest_content_inches = min(
    _caption_bottom_inches,
    float(leg_y + 0.015 * leg_h),
    float(TARGET_ROW_BOTTOMS[7])
)
_BOTTOM_MARGIN_INCHES = 0.14
_CROP_BOTTOM_INCHES = max(0.0, _lowest_content_inches - _BOTTOM_MARGIN_INCHES)
_CROP_BOTTOM_INCHES = min(_CROP_BOTTOM_INCHES, 1.10)
_NEW_HEIGHT_INCHES = FIG_H - _CROP_BOTTOM_INCHES
assert 4.20 <= _NEW_HEIGHT_INCHES <= FIG_H, (FIG_H, _NEW_HEIGHT_INCHES)
_crop_bbox = Bbox.from_bounds(
    0.0, _CROP_BOTTOM_INCHES, FIG_W, _NEW_HEIGHT_INCHES
)

base = 'FIG_H3_ENETSP_MATCHED_SOURCE_BOTTOM_CROPPED_PERCENT_CORRECT'
for suffix in ('png', 'pdf', 'svg'):
    path = OUT / f'{base}.{suffix}'
    fig.savefig(path, dpi=DPI if suffix == 'png' else None,
                facecolor='white', bbox_inches=_crop_bbox, pad_inches=0)
    if not path.is_file() or path.stat().st_size <= 500:
        raise RuntimeError(f'Export failed: {path}')
    print('SAVED', path, 'bytes', path.stat().st_size)
print(f'OUTPUT CANVAS: {FIG_W:.2f} x {_NEW_HEIGHT_INCHES:.3f} inches '
      f'(removed {_CROP_BOTTOM_INCHES:.3f} inches from bottom only)')
plt.close(fig)
try:
    from IPython.display import Image as IPyImage, display
    display(IPyImage(filename=str(OUT / f'{base}.png')))
except ImportError:
    pass


selected.to_csv(OUT/'MATCHED_H3_TRANSPORT_ENDPOINT_LABELS.csv', index=False)

# ---------------- Sensitivity / summaries ----------------
sweep = []
for sid,rowname,_ in TARGET_ROWS_IN:
    tru = TRUTH[sid]
    for method in ('Zoetrope','moscot','stVCR'):
        c = RESULT_CLASSIFICATIONS[(sid,method)]['classification']
        weights = RESULT_CLASSIFICATIONS[(sid,method)]['weights']
        for mult in PROXIMITY_RADIUS_SWEEP:
            near = c['on_raw'] & (c['nearest_expected_dist'] <= mult*tru['spacing'])
            evaluable = c['on_raw'] & ((c['n_known'] >= PROXIMITY_MIN_KNOWN_FOR_NEGATIVE) | near)
            sweep.append(dict(
                age=rowname, method=method, radius_multiple_5nn=mult,
                radius_physical=mult*tru['spacing'],
                near_fraction_all=fraction(near,weights),
                near_fraction_evaluable=(fraction(near,weights)/fraction(evaluable,weights)
                                         if fraction(evaluable,weights)>0 else np.nan),
                evaluable_fraction=fraction(evaluable,weights),
                observational_unit=('contour_length' if method=='Zoetrope' else 'cells')
            ))

for sid,rowname,_ in TARGET_ROWS_IN:
    tru = TRUTH[sid]
    c = REFERENCE_BACKGROUND[sid]
    weights = np.ones(len(c['on_raw']),float)
    for mult in PROXIMITY_RADIUS_SWEEP:
        near = c['on_raw'] & (c['nearest_expected_dist'] <= mult*tru['spacing'])
        evaluable = c['on_raw'] & ((c['n_known'] >= PROXIMITY_MIN_KNOWN_FOR_NEGATIVE) | near)
        sweep.append(dict(
            age=rowname, method='RAW_TISSUE_REFERENCE', radius_multiple_5nn=mult,
            radius_physical=mult*tru['spacing'],
            near_fraction_all=fraction(near,weights),
            near_fraction_evaluable=(fraction(near,weights)/fraction(evaluable,weights)
                                     if fraction(evaluable,weights)>0 else np.nan),
            evaluable_fraction=fraction(evaluable,weights),
            observational_unit='random_raw_tissue_cells'
        ))

pd.DataFrame(sweep).to_csv(OUT/'PROXIMITY_RADIUS_SENSITIVITY_1_TO_6_X5NN.csv', index=False)
summary = pd.DataFrame(rows)
summary.to_csv(OUT/'MATCHED_H3_SPATIAL_LABEL_SUMMARY.csv', index=False)

(OUT/'READ_ME_METHODS.json').write_text(json.dumps(dict(
    source='Exactly the same saved 6000 GW15 source identities for moscot/stVCR',
    selected='Only correct GW15 EN-ET-SP source cells physically near the FIVE contours',
    target='GW20 and GW22 H3_annotation beginning EN-ET-SP-',
    target_matching='EXPLORATORY spatial proximity: nearest EN-ET-SP annotated target cell within 4x target atlas median fifth-NN spacing, while remaining on raw tissue. Other target labels count as negative only if >=4 known H3 cells within 3x atlas 5NN. Original strict-majority 21NN score retained and a 1-6x radius sweep exported.',
    band_factor=BAND_RADIUS_FACTOR, n_sources=int(len(selected)),
    proximity_rule=dict(
        radius_factor=PROXIMITY_RADIUS_FACTOR,
        minimum_negative_known=PROXIMITY_MIN_KNOWN_FOR_NEGATIVE,
        sweep=PROXIMITY_RADIUS_SWEEP
    ),
    original_majority_audit=dict(
        k=TARGET_VOTE_K, radius_factor=TARGET_VOTE_RADIUS_FACTOR,
        min_known=TARGET_VOTE_MIN_KNOWN, threshold=TARGET_VOTE_THRESHOLD
    ),
    zoetrope='Five original GW15 Fisher contour levels advected to target ages, alongside individually advected selected source-cell z. Native target contours are the transported original levels (no new quantiling). Colors show observed target H3 label agreement; NOT per-cell XY trajectories.',
    baselines='moscot frozen barycentric conditional expectation and corrected stVCR frozen native XY endpoints',
    ages='GW20/GW22 are trained ages, NOT timepoint holdout',
    interpretation='Correct type = near observed EN-ET-SP region within a liberal 4x atlas fifth-NN tolerance, not independent fate truth. Incorrect type = not near when enough annotations exist. Unassigned = unassessed/outside raw tissue. Random-raw-tissue reference, original majority, and radius sensitivity are also saved.'
), indent=2))

print('\nRESULTS — EXPLORATORY H3 territorial proximity')
print(summary.to_string(index=False, float_format=lambda x:f'{x:.4g}'))
print('OUTPUT', OUT)

## Render IPC-SVZ-1 trajectories (Figure 6B)

In [ ]:
# Fresh kernel: NO S, maps, PLOT_CACHE, TRANSITIONS, or earlier cell required.
# Reconstructs the EXACT frozen H2 original quartile contours from saved Stage-1a
# and original saved H2 Fisher contour table (NOT a new experiment).
# Uses saved moscot conditional barycenters and verified corrected stVCR raw XY.

import os, json, hashlib
from pathlib import Path

import h5py
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.tri as mtri
import numpy as np
import pandas as pd
from matplotlib.collections import LineCollection
from matplotlib.gridspec import GridSpec
from matplotlib.patches import FancyBboxPatch
from scipy.spatial import cKDTree

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------
P = Path(os.environ.get(
    'SPATIAL_TI_PROJECT',
    os.environ["STP_PROJECT"]
))
D = Path(os.environ.get(
    'QIAN_DATA_ROOT',
    os.environ["QIAN_PFC_DATA_ROOT"]
))

RUN = P / 'experiments/qian_pfc_spatial_ti_all9_synthetic_defaults_v1/CPU_FULL'
H2ROOT = P / 'paper_figures/qian_PFC_H2_H3_Fisher_transition_matrices_TEXT_v1'
H2 = H2ROOT / 'H2_7_TRANSITIONS_MOSCOT_STVCR_ZOETROPE_v1'
OUT = P / 'paper_figures/qian_PFC_spatial_trajectory_H2_v4_PUBLICATION_LABELS_5p5x4p7_4X5NN'
OUT.mkdir(parents=True, exist_ok=True)

COHORT = H2 / 'FROZEN_SEVEN_H2_SOURCE_COHORT.csv.gz'
MOS = H2 / 'MOSCOT_H2_FROZEN_BARYCENTRIC_ENDPOINTS.csv.gz'
STV = H2 / 'STVCR_H2_FROZEN_NATIVE_SPATIAL_ENDPOINTS_UPDATED.csv.gz'
REC = H2 / 'STVCR_CORRECTED_NATIVE_EXPORT_VERIFIED_v1/stvcr_corrected_endpoint_export_receipt.json'
SCORES = H2ROOT / 'TOP10_H2_PHYSICAL_FISHER_CONTOURS_FIXED/TOP10_CONTOUR_PHYSICAL_BAND_SCORES.csv'
STAGE = RUN / 'tables/stage1a_all_cells.csv.gz'
ATLAS = D / 'merscope_integrated_855.h5ad'

SPEC = {
    1: ('GW15', 'UMB1117', 'F1b', 410492),
    2: ('GW20', 'FB080', 'F1', 665204),
}

for f in (
    COHORT, MOS, STV, REC, SCORES, STAGE, ATLAS,
    *[D / f'{v[0].lower()}.h5ad' for v in SPEC.values()]
):
    if not f.is_file():
        raise FileNotFoundError(f'Missing frozen H2 input: {f}')

rec = json.loads(REC.read_text())
qc = rec.get('target_coordinate_validation', [])
if rec.get('retrained') is not False or {r.get('age') for r in qc} != {'GW15', 'GW20', 'GW22'} \
   or any(r['p99_fraction_of_diameter'] > .001 for r in qc):
    raise RuntimeError('Corrected stVCR raw-target frame has not passed heldout coordinate verification')

if rec.get('h2_source_manifest_sha256') != hashlib.sha256(COHORT.read_bytes()).hexdigest():
    raise RuntimeError('Corrected stVCR receipt and frozen H2 cohort checksum differ')

# ------------------------------------------------------------
# STRICT SOURCE IDENTITY MATCH
# ------------------------------------------------------------
keys = ['transition', 'source_sid', 'target_sid', 'source_row',
        'source_key', 'source_label', 'target_label']

co = pd.read_csv(COHORT, dtype={'source_key': str})
mo = pd.read_csv(MOS, dtype={'source_key': str})
st = pd.read_csv(STV, dtype={'source_key': str})

if not (co[keys].equals(mo[keys]) and co[keys].equals(st[keys])):
    raise RuntimeError('moscot/stVCR source identity or ordering mismatch in saved tables')

select = co.transition.astype(int).eq(3).to_numpy()
assert select.sum() > 50
one = co.loc[select]

if not ((one.source_sid == 1) &
        (one.target_sid == 2) &
        (one.source_label == 'IPC-SVZ-1') &
        (one.target_label == 'EN-oSVZ-1')).all():
    raise RuntimeError('H2 transition #3 identity changed')

for table in (mo, st):
    arr = table.loc[select, ['x_pred_native', 'y_pred_native']].to_numpy(float)
    if not np.isfinite(arr).all():
        raise RuntimeError('Nonfinite baseline endpoints')

print('PASS frozen source cohort matched:', select.sum(), 'IPC-SVZ-1 cells')

# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------
def dec(v):
    return v.decode() if isinstance(v, (bytes, np.bytes_)) else str(v)

def rows(h, sample, region):
    o = h['obs']
    s = o['sample']
    r = o['region']
    names = [dec(v) for v in s['categories'][:]]
    regions = [dec(v) for v in r['categories'][:]]
    return np.flatnonzero(
        (s['codes'][:] == names.index(sample)) &
        (r['codes'][:] == regions.index(region))
    )

def strings(o, ix):
    if isinstance(o, h5py.Group) and 'codes' in o:
        cc = np.asarray(o['codes'][ix], dtype=int)
        cats = np.array([dec(x) for x in o['categories'][:]], dtype=object)
        return np.where(cc >= 0, cats[np.maximum(cc, 0)], '').astype(str)
    return np.array([dec(v) for v in o[ix]], dtype=str)

# ------------------------------------------------------------
# LOAD RAW SECTION XY / IDS
# ------------------------------------------------------------
S = {}
for sid, (age, sample, region, n) in SPEC.items():
    with h5py.File(D / f'{age.lower()}.h5ad', 'r') as f:
        ii = rows(f, sample, region)
        if len(ii) != n:
            raise RuntimeError(f'{age} raw section count changed')
        x = np.asarray(f['obsm']['spatial'][ii, :2], float)
        ids = strings(f['obs'][dec(f['obs'].attrs.get('_index', '_index'))], ii)
    S[sid] = {'xy': x, 'ids': ids, 'age': age}

# ------------------------------------------------------------
# RECONSTRUCT H2 LABELS BY ATLAS NN + 0.55*5NN SPACING
# ------------------------------------------------------------
with h5py.File(ATLAS, 'r') as f:
    for sid, (_, sample, region, n) in SPEC.items():
        ii = rows(f, sample, region)
        xyz = np.asarray(f['obsm']['spatial'][ii, :2], float)
        tree = cKDTree(xyz)
        probe = np.linspace(0, len(xyz) - 1, min(3000, len(xyz)), dtype=int)
        spacing = float(np.median(tree.query(xyz[probe], k=6)[0][:, 5]))
        d, nearest = tree.query(S[sid]['xy'], k=1, workers=-1)
        ok = d <= .55 * spacing
        labs = strings(f['obs']['H2_annotation'], ii)
        labels = np.full(n, '', dtype=object)
        labels[ok] = labs[nearest[ok]]
        if ok.mean() <= .90:
            raise RuntimeError(f'{S[sid]["age"]}: atlas matching below 90%')
        S[sid]['H2'] = labels
        print(S[sid]['age'], 'H2 annotation matched', f'{ok.mean():.2%}')

# ------------------------------------------------------------
# RECONSTRUCT STAGE-1A z FROM FROZEN TABLE
# ------------------------------------------------------------
header = set(pd.read_csv(STAGE, nrows=0).columns)
xycols = next((p for p in [('spatial_x', 'spatial_y'), ('x', 'y'), ('raw_x', 'raw_y')]
               if set(p) <= header), None)
idcol = next((c for c in ('original_cell_id', 'raw_cell_id', 'obs_name') if c in header), None)
use = ['slice_id', 'z_stage1a'] + list(xycols or []) + ([idcol] if idcol else [])

blocks = {sid: [] for sid in SPEC}
counts = np.zeros(9, dtype=np.int64)

for part in pd.read_csv(STAGE, usecols=use, chunksize=300000):
    ids = part.slice_id.to_numpy(int)
    counts += np.bincount(ids, minlength=9)
    for sid in SPEC:
        small = part.loc[ids == sid]
        if len(small):
            blocks[sid].append(small)

if counts.tolist() != [548567, 410492, 665204, 596754, 322648, 553738, 508417, 439986, 333113]:
    raise RuntimeError('Stage-1a source table no longer matches frozen nine-section panel')

for sid in SPEC:
    df = pd.concat(blocks[sid], ignore_index=True)
    z = df.z_stage1a.to_numpy(float)
    if len(z) != len(S[sid]['xy']) or not np.isfinite(z).all():
        raise RuntimeError('Wrong Stage-1a z table length')
    if xycols:
        err = np.linalg.norm(df[list(xycols)].to_numpy(float) - S[sid]['xy'], axis=1)
        if np.quantile(err, .99) > 1e-3:
            raise RuntimeError('Stage1 z/raw coordinate order mismatch')
    S[sid]['z'] = z

# ------------------------------------------------------------
# FROZEN H2 SOURCE/TARGET CONTOUR LEVELS
# ------------------------------------------------------------
sc = pd.read_csv(SCORES)
selected_score = sc.loc[sc['index'].astype(int).eq(3)]
if len(selected_score) != 1:
    raise RuntimeError('Original H2 contour score table has no unique transition #3')
r = selected_score.iloc[0]

if (r['source'], r['target'], r['interval']) != ('IPC-SVZ-1', 'EN-oSVZ-1', 'GW15_to_GW20'):
    raise RuntimeError('Saved Fisher contours belong to a different H2 transition')

source_levels = np.quantile(S[1]['z'][S[1]['H2'] == 'IPC-SVZ-1'], [.25, .5, .75])
saved_source = r[['src_z25', 'src_z50', 'src_z75']].to_numpy(float)
if not np.allclose(source_levels, saved_source, atol=2e-5, rtol=1e-6):
    raise RuntimeError('Recomputed source H2 quartiles disagree with the ORIGINAL frozen plot')

target_levels = r[['fisher_z25', 'fisher_z50', 'fisher_z75']].to_numpy(float)
if not np.isfinite(target_levels).all() or not np.all(np.diff(target_levels) > 0):
    raise RuntimeError('Invalid saved original Fisher transported quartiles')

print('PASS original H2 #3 Fisher contour levels reproduced')

# ------------------------------------------------------------
# ORIGINAL MASKED DELAUNAY FOR SOURCE / TARGET
# ------------------------------------------------------------
rng = np.random.default_rng(20261030)
mesh = {}
for sid in (1, 2):
    xy = S[sid]['xy']
    z = S[sid]['z']
    ix = rng.choice(len(z), size=min(24000, len(z)), replace=False)
    pts = xy[ix]
    keep = np.unique(np.round(pts, 6), axis=0, return_index=True)[1]
    pts = pts[keep]
    ztri = z[ix[keep]]
    triang = mtri.Triangulation(pts[:, 0], pts[:, 1])
    tri = triang.triangles
    a, b, c = pts[tri[:, 0]], pts[tri[:, 1]], pts[tri[:, 2]]
    dist = cKDTree(pts).query(pts, k=6)[0][:, 5]
    limit = 3.5 * np.maximum.reduce([dist[tri[:, 0]], dist[tri[:, 1]], dist[tri[:, 2]]])
    long = np.maximum.reduce([
        np.linalg.norm(a - b, axis=1),
        np.linalg.norm(b - c, axis=1),
        np.linalg.norm(c - a, axis=1)
    ])
    triang.set_mask(long > limit)
    mesh[sid] = (triang, ztri)
    _ = rng.choice(len(z), size=min(48000, len(z)), replace=False)

# ------------------------------------------------------------
# EVALUATION / DRAW HELPERS
# ------------------------------------------------------------
RED = '#D62728'
BLACK = '#1F2428'
CYAN = '#00A6BD'
TARGET_GREEN = '#8BD39E'
GRAY = '#969EA6'
UNKNOWN = '#A1A6AC'
INK = '#26343D'

BAND_RADIUS_FACTOR = 2.5
PROXIMITY_RADIUS_FACTOR = 4.0
PROXIMITY_MIN_KNOWN_FOR_NEGATIVE = 4
PROXIMITY_RADIUS_SWEEP = (1.0, 2.0, 3.0, 4.0, 6.0)
TARGET_VOTE_K = 21
TARGET_VOTE_RADIUS_FACTOR = 3.0
TARGET_VOTE_MIN_KNOWN = 7
TARGET_VOTE_THRESHOLD = 0.50
RAW_COVERAGE_FACTOR = 1.50
SEED_DRAW = 20261103
DPI = 450

mpl.rcParams.update({
    'font.family': 'DejaVu Sans',
    'font.size': 9.2,
    'axes.titlesize': 10.2,
    'axes.titleweight': 'bold',
    'axes.labelsize': 9.1,
    'xtick.labelsize': 8.7,
    'ytick.labelsize': 8.7,
    'axes.edgecolor': INK,
    'axes.labelcolor': INK,
    'text.color': INK,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'svg.fonttype': 'none',
    'figure.facecolor': 'white',
    'savefig.facecolor': 'white',
})

def local_5nn(xy, k=6):
    xy = np.asarray(xy, float)
    ix = np.linspace(0, len(xy) - 1, min(3000, len(xy)), dtype=int)
    return float(np.median(cKDTree(xy).query(xy[ix], k=k, workers=-1)[0][:, 5]))

def contour_paths(tri, zvals, levels):
    vals = np.asarray(levels, float)
    vals = np.unique(np.sort(vals[(vals > np.min(zvals)) & (vals < np.max(zvals))]))
    if not len(vals):
        raise RuntimeError('No valid contour levels within saved z triangulation')
    temp, ax = plt.subplots(figsize=(1, 1))
    try:
        cs = ax.tricontour(tri, zvals, levels=vals, colors='k')
        paths = [np.asarray(p, float) for family in cs.allsegs for p in family
                 if len(p) >= 2 and np.isfinite(p).all()]
    finally:
        plt.close(temp)
    if not paths:
        raise RuntimeError('Triangulation produced no finite contour paths')
    return paths

def segments_at_resolution(paths, step):
    if not np.isfinite(step) or step <= 0:
        raise ValueError('Bad contour sampling step')
    pieces = []
    for path in paths:
        diff = np.diff(path, axis=0)
        k = np.maximum(1, np.ceil(np.linalg.norm(diff, axis=1) / step).astype(np.int32))
        start = np.repeat(path[:-1], k, axis=0)
        dr = np.repeat(diff, k, axis=0)
        accum = np.cumsum(k)
        j = np.arange(int(accum[-1])) - np.repeat(accum - k, k)
        frac = j / np.repeat(k, k)
        delta = dr / np.repeat(k, k)[:, None]
        p0 = start + frac[:, None] * dr
        pieces.append(np.stack([p0, p0 + delta], axis=1))
    out = np.concatenate(pieces)
    if not len(out):
        raise RuntimeError('Empty source contour line segments')
    return out

def contour_touched(source_xy, paths, radius):
    segs = segments_at_resolution(paths, step=radius / 3.0)
    mid = segs.mean(axis=1)
    distances, _ = cKDTree(mid).query(source_xy, k=1, workers=-1)
    selected = np.isfinite(distances) & (distances <= radius)
    return selected, distances

def make_truth(atlas, target_sid, spec, label_key, acceptable, raw_tissue_xy):
    age, sample, region = spec[:3]
    with h5py.File(atlas, 'r') as h:
        ii = rows(h, sample, region)
        xy = np.asarray(h['obsm']['spatial'][ii, :2], np.float64)
        labels = strings(h['obs'][label_key], ii)
    ok = np.array([bool(acceptable(v)) for v in labels], dtype=bool)
    missing = {'', 'nan', 'none', 'na', 'n/a', 'unknown', 'unassigned', 'unannotated', 'unlabeled'}
    known = np.array([str(v).strip().lower() not in missing for v in labels], dtype=bool)
    if not np.any(ok):
        raise RuntimeError(f'No expected H2 domain in target sid={target_sid}')
    spacing = local_5nn(xy)
    raw_tissue_xy = np.asarray(raw_tissue_xy, dtype=np.float64)
    raw_spacing = local_5nn(raw_tissue_xy)
    return dict(
        xy=xy,
        labels=labels,
        good=ok,
        label_known=known,
        tree=cKDTree(xy),
        good_tree=cKDTree(xy[ok]),
        proximity_radius=PROXIMITY_RADIUS_FACTOR * spacing,
        min_known_negative=PROXIMITY_MIN_KNOWN_FOR_NEGATIVE,
        vote_radius=TARGET_VOTE_RADIUS_FACTOR * spacing,
        vote_k=TARGET_VOTE_K,
        min_known=TARGET_VOTE_MIN_KNOWN,
        vote_threshold=TARGET_VOTE_THRESHOLD,
        spacing=spacing,
        raw_tree=cKDTree(raw_tissue_xy),
        raw_spacing=raw_spacing,
        raw_max_dist=RAW_COVERAGE_FACTOR * raw_spacing,
        n_good=int(ok.sum()),
        sid=target_sid
    )

def classify_region_support(xy, truth):
    xy = np.asarray(xy, float)
    d, ix = truth['tree'].query(
        xy, k=truth['vote_k'],
        distance_upper_bound=truth['vote_radius'],
        workers=-1
    )
    d = np.asarray(d).reshape(len(xy), -1)
    ix = np.asarray(ix).reshape(len(xy), -1)
    valid = np.isfinite(d) & (ix < truth['tree'].n)
    safe_ix = np.where(valid, ix, 0)
    known_votes = valid & truth['label_known'][safe_ix]
    correct_votes = known_votes & truth['good'][safe_ix]
    n_known = known_votes.sum(axis=1)
    n_correct = correct_votes.sum(axis=1)
    vote_fraction = np.divide(
        n_correct, n_known,
        out=np.full(len(xy), np.nan),
        where=n_known > 0
    )
    raw_dist, _ = truth['raw_tree'].query(xy, k=1, workers=-1)
    on_raw = np.isfinite(raw_dist) & (raw_dist <= truth['raw_max_dist'])
    nearest_expected_dist, _ = truth['good_tree'].query(xy, k=1, workers=-1)
    near_expected = (nearest_expected_dist <= truth['proximity_radius'])
    strict_majority = (
        on_raw & (n_known >= truth['min_known']) &
        (vote_fraction > truth['vote_threshold'])
    )
    correct = on_raw & near_expected
    annotated = on_raw & ((n_known >= truth['min_known_negative']) | near_expected)
    wrong = annotated & ~correct
    unresolved = on_raw & ~annotated
    outside = ~on_raw
    labels = np.full(len(xy), 'INSUFFICIENT_H2_COVERAGE', dtype=object)
    labels[correct] = 'NEAR_EN_oSVZ_1_TERRITORY'
    labels[wrong] = 'NO_EXPECTED_H2_WITHIN_PROXIMITY_RADIUS'
    labels[outside] = 'OUTSIDE_RAW_TISSUE_COVERAGE'
    status = np.full(len(xy), 'unresolved', dtype=object)
    status[correct] = 'near_expected_H2'
    status[wrong] = 'not_near_expected_H2'
    status[outside] = 'outside_raw_coverage'
    return dict(
        correct=correct, wrong=wrong, unresolved=unresolved, outside=outside,
        on_raw=on_raw, annotated=annotated, raw_dist=raw_dist,
        n_known=n_known, n_correct_votes=n_correct, vote_fraction=vote_fraction,
        nearest_expected_dist=nearest_expected_dist,
        nearest_expected_dist_in_5nn=nearest_expected_dist / truth['spacing'],
        strict_majority=strict_majority, labels=np.asarray(labels, dtype=str),
        status=np.asarray(status, dtype=str)
    )

def fraction(mask, weights):
    weights = np.asarray(weights, dtype=float)
    mask = np.asarray(mask, bool)
    return float(weights[mask].sum() / weights.sum()) if weights.sum() > 0 else float('nan')

def region_summary(c, weights):
    covered = c['annotated']
    w = np.asarray(weights, float)
    covered_fraction = fraction(covered, w)
    return dict(
        segment_support_fraction=fraction(c['correct'], w),
        correct_given_annotated_fraction=(
            fraction(c['correct'], w) / covered_fraction if covered_fraction > 0 else float('nan')
        ),
        annotated_fraction=covered_fraction,
        incorrect_label_fraction=fraction(c['wrong'], w),
        atlas_unresolved_fraction=fraction(c['unresolved'], w),
        off_tissue_fraction=fraction(c['outside'], w),
    )

def colored_contour_segments(paths, truth):
    segments = segments_at_resolution(paths, truth['spacing'] * .85)
    classification = classify_region_support(segments.mean(axis=1), truth)
    lengths = np.linalg.norm(segments[:, 1] - segments[:, 0], axis=1)
    colors = np.full(len(segments), UNKNOWN, dtype=object)
    colors[classification['correct']] = RED
    colors[classification['wrong']] = BLACK
    summary = region_summary(classification, lengths)
    summary.update(
        n_segments=len(segments),
        n_evaluable=int(classification['annotated'].sum()),
        segments=segments,
        segment_lengths=lengths,
        classification=classification
    )
    return segments, colors, summary

# ------------------------------------------------------------
# UPDATED SOURCE DRAW: CONTOURS ON TOP AND VISIBLE
# ------------------------------------------------------------
def source_draw(ax, xy, ref_xy, selected_xy, paths, bounds, src_label):
    lo, hi = bounds

    # background tissue
    ax.scatter(
        xy[:, 0], xy[:, 1],
        s=.24, c=GRAY, alpha=.23,
        linewidths=0, rasterized=True, zorder=1
    )

    # observed source label population
    if len(ref_xy):
        ax.scatter(
            ref_xy[:, 0], ref_xy[:, 1],
            s=.48, c=CYAN, alpha=.56,
            linewidths=0, rasterized=True, zorder=2
        )

    # selected source cells touching the contour band
    ax.scatter(
        selected_xy[:, 0], selected_xy[:, 1],
        s=3.4, c=CYAN, alpha=.90,
        linewidths=0, rasterized=True, zorder=3
    )

    # draw contours LAST so they do not disappear under the cyan points
    # white halo + dark underlay + red foreground makes them visible
    for path in paths:
        ax.plot(path[:, 0], path[:, 1],
                color='white', lw=2.0, alpha=.96, zorder=6)
        ax.plot(path[:, 0], path[:, 1],
                color=BLACK, lw=1.15, alpha=.72, zorder=7)
        ax.plot(path[:, 0], path[:, 1],
                color=RED, lw=.92, alpha=.95, zorder=8)

    ax.set_xlim(lo[0], hi[0])
    ax.set_ylim(hi[1], lo[1])
    ax.set_aspect('equal')
    ax.axis('off')

def target_draw(ax, background, reference, bounds, truth, point_xy=None, ti_paths=None):
    lo, hi = bounds
    ax.scatter(
        background[:, 0], background[:, 1],
        s=.25, c=GRAY, alpha=.18,
        linewidths=0, rasterized=True, zorder=1
    )
    if len(reference):
        ax.scatter(
            reference[:, 0], reference[:, 1],
            s=.56, c=TARGET_GREEN, alpha=.78,
            linewidths=0, rasterized=True, zorder=2
        )

    info = {}
    if ti_paths is not None:
        seg, col, info = colored_contour_segments(ti_paths, truth)
        ax.add_collection(LineCollection(
            seg, colors=col, linewidths=1.13,
            alpha=.96, zorder=6, rasterized=False
        ))
    elif point_xy is not None:
        c = classify_region_support(point_xy, truth)
        for mask, color, s, alpha in (
            (c['unresolved'] | c['outside'], UNKNOWN, 3.6, .69),
            (c['wrong'], BLACK, 3.9, .90),
            (c['correct'], RED, 4.2, .91),
        ):
            if mask.any():
                ax.scatter(
                    point_xy[mask, 0], point_xy[mask, 1],
                    s=s, c=color, alpha=alpha,
                    linewidths=0, zorder=6, rasterized=True
                )
        summary = region_summary(c, np.ones(len(point_xy), float))
        info = dict(
            n=len(point_xy),
            n_correct=int(c['correct'].sum()),
            fraction_correct=float(c['correct'].mean()),
            correct=c['correct'],
            inside=c['on_raw'],
            status=c['status'],
            raw_dist=c['raw_dist'],
            n_known=c['n_known'],
            n_correct_votes=c['n_correct_votes'],
            vote_fraction=c['vote_fraction'],
            nearest_expected_dist=c['nearest_expected_dist'],
            nearest_expected_dist_in_5nn=c['nearest_expected_dist_in_5nn'],
            strict_majority=c['strict_majority'],
            classification=c,
            **summary
        )
    else:
        raise ValueError('Provide precisely one prediction type')

    ax.set_xlim(lo[0], hi[0])
    ax.set_ylim(hi[1], lo[1])
    ax.set_aspect('equal')
    ax.axis('off')
    return info

def fixed_background(xy, reference, sid, max_bg=24000, max_ref=18000):
    rng = np.random.default_rng(SEED_DRAW + sid)
    bi = np.sort(rng.choice(len(xy), size=min(max_bg, len(xy)), replace=False))
    ri = np.sort(rng.choice(len(reference), size=min(max_ref, len(reference)), replace=False))
    pad = np.maximum(np.ptp(xy, axis=0) * .028, 1e-5)
    return xy[bi], reference[ri], (xy.min(0) - pad, xy.max(0) + pad)

def safe_export(fig, out, base):
    for suffix in ('png', 'pdf', 'svg'):
        p = out / f'{base}.{suffix}'
        fig.savefig(
            p,
            dpi=DPI if suffix == 'png' else None,
            facecolor='white',
            bbox_inches=None,
            pad_inches=0
        )
        assert p.is_file() and p.stat().st_size > 500, f'Figure failed to save: {p}'
        print('SAVED', p, 'size', p.stat().st_size, 'bytes')

# ------------------------------------------------------------
# H2 TRUTH + MATCHED SOURCE BAND
# ------------------------------------------------------------
true = make_truth(
    ATLAS, 2, SPEC[2], 'H2_annotation',
    lambda s: str(s) == 'EN-oSVZ-1',
    S[2]['xy']
)

source_idxs = one.source_row.to_numpy(np.int64)
source_xy = S[1]['xy'][source_idxs]
source_z = S[1]['z'][source_idxs]
assert (np.asarray(S[1]['H2'])[source_idxs] == 'IPC-SVZ-1').all(), \
    'Frozen H2 source cohort no longer maps to IPC-SVZ-1 labels'

source_paths = contour_paths(*mesh[1], source_levels)
source_spacing = local_5nn(S[1]['xy'])
source_radius = BAND_RADIUS_FACTOR * source_spacing
touched, distances = contour_touched(source_xy, source_paths, source_radius)
if touched.sum() < 15:
    raise RuntimeError(f'Only {touched.sum()} H2 shared source cells are within original contour band')

print('H2 contour-touching shared cohort:', int(touched.sum()), '/', len(one), 'radius', source_radius)

selected = one.loc[touched].reset_index(drop=True).copy()
selected['distance_to_contour'] = distances[touched]

# ------------------------------------------------------------
# ZOETROPE FLOW FOR GW20 TARGET CONTOURS
# ------------------------------------------------------------
import importlib.util
import sys
import torch
from scipy.interpolate import RegularGridInterpolator

REPO = Path(os.environ.get(
    'FLOW_STP_REPO',
    os.environ["FLOW_STP_REPO"]
))
if not (REPO / 'src/flow_stp/stage1/stage1a.py').is_file():
    REPO = P

SRC = REPO / 'src/flow_stp/stage1/stage1a.py'
CKPT = RUN / 'model_final.pt'
if not SRC.is_file() or not CKPT.is_file():
    raise FileNotFoundError('Zoetrope model/Stage1 source needed for per-cell z-flow')

ck = torch.load(CKPT, map_location='cpu', weights_only=False)
if ck.get('method_version') != 'stage1a_shared_logE_global_gauge_v1':
    raise RuntimeError('Unexpected Zoetrope checkpoint version')

spec_mod = importlib.util.spec_from_file_location('pfc_h2_matched_flow_stage1', SRC)
module = importlib.util.module_from_spec(spec_mod)
sys.modules[spec_mod.name] = module
spec_mod.loader.exec_module(module)

conf = {k: v for k, v in ck['model_config'].items()
        if k in module.FlowSTPConfig.__dataclass_fields__}
model = module.FlowSTPModel(module.FlowSTPConfig(**conf)).cpu().eval()
model.load_state_dict(ck['model_state_dict'], strict=True)
torch.set_num_threads(min(6, os.cpu_count() or 2))

zmin = min(S[1]['z'].min(), S[2]['z'].min()) - .5
zmax = max(S[1]['z'].max(), S[2]['z'].max()) + .5
zg = np.linspace(zmin, zmax, 301)
tg = np.linspace(0, 5/7, 96)
vfield = np.empty((len(tg), len(zg)), float)

for t_i, t in enumerate(tg):
    for zi in range(0, len(zg), 384):
        zz = torch.as_tensor(zg[zi:zi+384], dtype=torch.float32)
        tt = torch.full_like(zz, float(t))
        _, dz = torch.func.jvp(model.fisher_flat, (zz, tt),
                               (torch.ones_like(zz), torch.zeros_like(tt)))
        _, dt = torch.func.jvp(model.fisher_flat, (zz, tt),
                               (torch.zeros_like(zz), torch.ones_like(tt)))
        a = (-((dz * dt).sum(1) / (dz.square().sum(1) + 1e-8))).detach().numpy()
        vfield[t_i, zi:zi+len(a)] = a

V = RegularGridInterpolator((tg, zg), vfield, bounds_error=False, fill_value=np.nan)

def advect_selected(y0):
    y = np.asarray(y0, float).copy()
    n = 240
    h = (5/7) / n
    def f(t, v):
        return V(np.column_stack((np.full(len(v), t), v)))
    for j in range(n):
        t = j * h
        k1 = f(t, y)
        k2 = f(t + h/2, y + k1*h/2)
        k3 = f(t + h/2, y + k2*h/2)
        k4 = f(t + h, y + k3*h)
        y += h * (k1 + 2*k2 + 2*k3 + k4) / 6
    if not np.isfinite(y).all():
        raise RuntimeError('Selected cells leave native Fisher grid')
    return y

selected['zoetrope_z_GW20'] = advect_selected(source_z[touched])

check = advect_selected(source_levels)
print('H2 quartile z comparison; saved versus integrated:',
      list(zip(np.round(target_levels, 5), np.round(check, 5))))
if np.max(np.abs(check - target_levels)) > .05:
    raise RuntimeError('Saved H2 contour flow disagrees strongly with this checkpoint')

del model, ck

# ------------------------------------------------------------
# MATCH PREDICTIONS
# ------------------------------------------------------------
mos_xy = mo.loc[select, ['x_pred_native', 'y_pred_native']].to_numpy(float)[touched]
stv_xy = st.loc[select, ['x_pred_native', 'y_pred_native']].to_numpy(float)[touched]
assert len(mos_xy) == len(stv_xy) == len(selected)

PLOTS = {}
for sid, label in ((1, 'IPC-SVZ-1'), (2, 'EN-oSVZ-1')):
    xy = S[sid]['xy']
    if sid == 1:
        ref = xy[np.asarray(S[1]['H2']) == label]
    else:
        ref = true['xy'][true['good']]
    PLOTS[sid] = fixed_background(xy, ref, sid)

raw_reference = classify_region_support(PLOTS[2][0], true)
print('GW20 random raw-tissue proximity reference:',
      f'{100*raw_reference["correct"].mean():.1f}%')

selected['near_source_contour'] = True
selected.to_csv(OUT / 'MATCHED_H2_CONTOUR_SOURCE_CELLS.csv', index=False)

# ------------------------------------------------------------
# FINAL FIGURE
# ------------------------------------------------------------
FIG_W, FIG_H = 5.50, 4.70
METHODS = ('Zoetrope', 'moscot', 'stVCR')
METHOD_X = (1.57, 2.86, 4.15)
SLOT_W = 1.26

def _ratio(bounds):
    low, high = bounds
    width = float(high[0] - low[0])
    height = float(high[1] - low[1])
    if width <= 0 or height <= 0:
        raise RuntimeError('Degenerate tissue geometry')
    return height / width

fig = plt.figure(figsize=(FIG_W, FIG_H), facecolor='white')

fig.text(.5, 4.54/FIG_H, 'IPC to migratory neuron transition',
         ha='center', va='center', fontsize=11.0, fontweight='bold', color=INK)

for i, meth in enumerate(METHODS):
    fig.text((METHOD_X[i] + SLOT_W/2) / FIG_W, 4.075 / FIG_H, meth,
             ha='center', va='center', fontsize=10.0, fontweight='bold', color=INK)

ratio_target = _ratio(PLOTS[2][2])
TARGET_TOP = 3.86
TARGET_MIN_BOTTOM = .64
TGT_W = min(SLOT_W, (TARGET_TOP - TARGET_MIN_BOTTOM) / ratio_target)
TGT_H = TGT_W * ratio_target
TGT_BOTTOM = TARGET_TOP - TGT_H
TGT_CY = TGT_BOTTOM + .5 * TGT_H

SRC_CX = .72
SRC_TOP = 3.67
ratio_source = _ratio(PLOTS[1][2])
SRC_W = min(1.26, 1.02 / ratio_source)
SRC_H = SRC_W * ratio_source
SRC_BOTTOM = SRC_TOP - SRC_H

bg, ref, bounds = PLOTS[1]
ax_source = fig.add_axes([
    (SRC_CX - SRC_W/2) / FIG_W,
    SRC_BOTTOM / FIG_H,
    SRC_W / FIG_W,
    SRC_H / FIG_H
])
source_draw(ax_source, bg, ref, source_xy[touched], source_paths, bounds, 'IPC-SVZ-1')

fig.text(SRC_CX / FIG_W, (SRC_TOP + .12) / FIG_H, 'GW15 · IPC-SVZ-1',
         ha='center', va='bottom', fontsize=9.8, fontweight='bold', color=INK)
fig.text(SRC_CX / FIG_W, (SRC_BOTTOM - .10) / FIG_H,
         f'{len(selected):,} source cells',
         ha='center', va='top', fontsize=8.4, color=INK)

leg_w, leg_h = 1.40, 1.42
leg_top = SRC_BOTTOM - .28
leg_bottom = leg_top - leg_h
leg_ax = fig.add_axes([
    (SRC_CX - leg_w/2) / FIG_W,
    leg_bottom / FIG_H,
    leg_w / FIG_W,
    leg_h / FIG_H
])
leg_ax.axis('off')
leg_ax.add_patch(FancyBboxPatch(
    (.018, .015), .964, .970,
    boxstyle='round,pad=0.004,rounding_size=0.026',
    transform=leg_ax.transAxes,
    edgecolor='#C8CFD5',
    facecolor='white',
    linewidth=.72
))

LEGEND_ITEMS = (
    ('IPC-SVZ-1 source', CYAN),
    ('EN-oSVZ-1 target', TARGET_GREEN),
    ('Correct type', RED),
    ('Incorrect type', BLACK),
    ('Unassigned', UNKNOWN),
)
for i, (name, color) in enumerate(LEGEND_ITEMS):
    y = .855 - .175 * i
    leg_ax.scatter([.105], [y], s=30, c=color, linewidths=0,
                   transform=leg_ax.transAxes, zorder=3)
    leg_ax.text(.190, y, name, ha='left', va='center',
                fontsize=8.2, color=INK, transform=leg_ax.transAxes)

fig.text(1.45/FIG_W, TGT_CY/FIG_H, 'GW20',
         ha='center', va='center', rotation=90,
         fontsize=9.8, fontweight='bold', color=INK)

target_paths = contour_paths(*mesh[2], target_levels)
bg, ref, bounds = PLOTS[2]

results = []
for i, method in enumerate(METHODS):
    slot_left = METHOD_X[i]
    ax = fig.add_axes([
        (slot_left + (SLOT_W - TGT_W)/2) / FIG_W,
        TGT_BOTTOM / FIG_H,
        TGT_W / FIG_W,
        TGT_H / FIG_H
    ])
    if method == 'Zoetrope':
        stat = target_draw(ax, bg, ref, bounds, true, ti_paths=target_paths)
        val = float(stat['segment_support_fraction'])
    else:
        xy = mos_xy if method == 'moscot' else stv_xy
        stat = target_draw(ax, bg, ref, bounds, true, point_xy=xy)
        val = float(stat['fraction_correct'])

    results.append(dict(
        age='GW20',
        method=method,
        n_source=len(selected),
        percent_correct_display=100 * val,
        annotation_coverage_fraction=stat['annotated_fraction'],
        unknown_atlas_fraction=stat['atlas_unresolved_fraction'],
        off_raw_tissue_fraction=stat['off_tissue_fraction'],
    ))

    fig.text((slot_left + SLOT_W/2) / FIG_W, (TGT_BOTTOM - .12) / FIG_H,
             f'{100*val:.0f}% correct',
             ha='center', va='top',
             fontsize=9.0, fontweight='bold', color=INK)

fig_basename = 'FIG_H2_IPCSVZ1_ENOSVZ1_PUBLICATION_5p5W_4p7H_4X5NN_LABELS'
safe_export(fig, OUT, fig_basename)

print(f'FIGURE: {FIG_W:.2f} x {FIG_H:.2f} inches, Zoetrope/moscot/stVCR')
plt.show()
plt.close(fig)

selected.to_csv(OUT / 'MATCHED_H2_TRANSPORT_ENDPOINT_LABELS.csv', index=False)
pd.DataFrame(results).to_csv(OUT / 'MATCHED_H2_SPATIAL_LABEL_SUMMARY.csv', index=False)

(OUT / 'READ_ME_METHODS.json').write_text(json.dumps({
    'transition': 'GW15 IPC-SVZ-1 -> GW20 EN-oSVZ-1',
    'label': 'H2_annotation',
    'source_cohort': 'Saved transition #3, identical source cells for both endpoint methods',
    'selection': 'Subset within 2.5 x source-tissue 5NN of original 3 source quartiles',
    'zoetrope': '3 ORIGINAL source H2 quartile levels transported by fitted Fisher field; displays contour length support, not individual native XY endpoints',
    'comparators': 'Frozen moscot barycentric and corrected stVCR native physical endpoints',
    'scoring': '4 x atlas fifth-NN near-positive proximity; on raw physical tissue; strict 21NN majority separately audited',
    'evaluation': 'Zoetrope contour length; other methods same selected source cells, cell counts',
    'caveat': 'GW20 is a model-training age in the spatial trajectory experiment',
    'figure_inches': [5.50, 4.70],
}, indent=2))

print(pd.DataFrame(results).to_string(index=False))
print('OUTPUT:', OUT)

## Optional upstream EN-ET-SP cohort and moscot export

Run this section by itself after setup only to reconstruct upstream exports. Use QIAN_BARY_PHASE=figure for moscot barycenters or stvcr_export in the original stVCR kernel for its pre-correction export. Set ZOETROPE_EXPORT_PFC_SPATIAL=1 before running this cell.

In [ ]:
from __future__ import annotations
if os.environ.get("ZOETROPE_EXPORT_PFC_SPATIAL", "0") == "1":
    """Qian PFC | GW15 EN-ET-SP domain -> GW20/GW22 spatial trajectories.

    RUNS FROM THE FOUR USER-SUPPLIED PFC TRAINING NOTEBOOKS:
    Spatial TI: qian_pfc_spatial_ti_all9_synthetic_defaults_v1/CPU_FULL
    moscot: moscot_slices_1_2_7_AUTHOR_TUTORIAL_GPU
    stVCR: stvcr_slices_1_2_7_official
    STORIES: stories_slices_1_2_7_tutorial_GPU (provenance only; no physical XY forecast).

    PHASES (same cell, different kernel):
      os.environ['QIAN_BARY_PHASE']='stvcr_export'  # run in original stVCR training kernel
      os.environ['QIAN_BARY_PHASE']='figure'        # run in Spatial TI CPU kernel

    Read-only trained checkpoints; new H3-derived source cohort and new TI contours;
    new 6,000-cell stVCR ODE trajectories (not the older cached points);
    exact low-rank two-hop conditional moscot barycentric expectation.
    GW20 and GW22 are TRAINING ages, NOT held-out. H3 is expression-domain
    reference, not ground-truth lineage. Different model objects are compared visually.
    """
    import os, sys, json, hashlib, importlib, importlib.util, gc
    from pathlib import Path
    import numpy as np
    import pandas as pd
    import h5py
    import matplotlib.pyplot as plt
    import matplotlib.tri as mtri
    import torch
    from matplotlib.lines import Line2D
    from matplotlib.gridspec import GridSpec
    from scipy.interpolate import RegularGridInterpolator
    from scipy.spatial import cKDTree

    PHASE=os.environ.get('QIAN_BARY_PHASE','figure').lower()
    assert PHASE in ('stvcr_export','figure'), PHASE
    PROJECT=Path(os.environ.get('SPATIAL_TI_PROJECT', os.environ["STP_PROJECT"]))
    DATA=Path(os.environ.get('QIAN_DATA_ROOT',os.environ["QIAN_PFC_DATA_ROOT"]))
    REPO=Path(os.environ.get('FLOW_STP_REPO',os.environ["FLOW_STP_REPO"]))
    if not (REPO/'src/flow_stp/stage1/stage1a.py').is_file():REPO=PROJECT
    TI_SOURCE=REPO/'src/flow_stp/stage1/stage1a.py'
    TI_RUN=PROJECT/'experiments/qian_pfc_spatial_ti_all9_synthetic_defaults_v1/CPU_FULL'
    ROOT=PROJECT/'experiments/qian_pfc_publication_4way_20261024'
    MOS=ROOT/'moscot_slices_1_2_7_AUTHOR_TUTORIAL_GPU'
    STV=ROOT/'stvcr_slices_1_2_7_official'
    STORIES=ROOT/'stories_slices_1_2_7_tutorial_GPU'
    ATLAS=DATA/'merscope_integrated_855.h5ad'
    OUT=PROJECT/'paper_figures/qian_ENETSP_spatial_transport_LATEST_3AGE_BARYCENTRIC_v1'
    OUT.mkdir(parents=True,exist_ok=True)
    STV_CSV={2:OUT/'stVCR_NEW3AGE_GW15_to_GW20_6000.csv.gz',
             7:OUT/'stVCR_NEW3AGE_GW15_to_GW22_6000.csv.gz'}
    STV_RECEIPT=OUT/'stVCR_NEW3AGE_export_receipt.json'
    COHORT=OUT/'GW15_ENETSP_source_cells_REDERIVED.csv.gz'
    SPECS={1:('GW15','UMB1117','F1b',410492,0.),
           2:('GW20','FB080','F1',665204,5/7),
           7:('GW22','FB123','F2',439986,1.)}
    SEED=20261022
    SOURCE_MATCH_RADIUS_FACTOR=0.55
    N_SV=6000
    BARY_MOMENT_BLOCK=20000
    BARY_Q_BLOCK=12000
    N_CONTOURS=5
    MESH_MAX=23500
    MESH_EDGE_FACTOR=4.3
    BACKGROUND_MAX=28000
    REFERENCE_MAX=25000
    N_FIELD_TIME,N_FIELD_Z=121,241
    FIELD_BATCH=768
    PLOT_RED='#d53327'; OBS_CYAN='#00a6bd';BG_GRAY='#949aa3'
    DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    def need(p):
        p=Path(p)
        if not p.is_file():raise FileNotFoundError(f'Missing required file from NEW training notebooks: {p}')
        return p

    def sha(p):
        h=hashlib.sha256()
        with open(need(p),'rb') as f:
            for b in iter(lambda:f.read(4<<20),b''):h.update(b)
        return h.hexdigest()

    required=[ATLAS,STV/'SUCCESS.json',STV/'stvcr_preprocessed_3tissue.h5ad',
              STV/'stvcr_model_full.pt',STV/'stvcr_rigid_transform.pt',
              MOS/'SUCCESS.json',MOS/'moscot_input_3sections.h5ad',
              TI_RUN/'model_final.pt',TI_RUN/'FIT_VERIFIED.json',TI_SOURCE]
    required += [DATA/f'{age.lower()}.h5ad' for age,_,_,_,_ in SPECS.values()]
    for a,b in ((1,2),(2,7)):
        for part in ('Q.npy','R.npy','g.npy','source_obs_positions.npy','target_obs_positions.npy','solution_manifest.json'):
            required.append(MOS/('GW15_to_GW20' if a==1 else 'GW20_to_GW22')/part)
    for p in required:need(p)
    for method,path in [('Spatial TI',TI_RUN/'FIT_VERIFIED.json'),('moscot',MOS/'SUCCESS.json'),
                        ('stVCR',STV/'SUCCESS.json')]:
        status=json.loads(path.read_text())
        if status.get('smoke',False):raise RuntimeError(method+' is a smoke run')
        print('NEW training:',method,'--',path.parent,flush=True)
    print('Phase:',PHASE,'| device:',DEVICE,'| output:',OUT,flush=True)

    def decode(v):return v.decode('utf-8') if isinstance(v,(bytes,np.bytes_)) else str(v)
    def section_rows(h,sample,region):
        o=h['obs'];ss=o['sample'];rr=o['region']
        sc=[decode(v) for v in ss['categories'][:]];rc=[decode(v) for v in rr['categories'][:]]
        if sample not in sc or region not in rc:raise RuntimeError('Missing '+sample+'/'+region)
        return np.flatnonzero((ss['codes'][:]==sc.index(sample))&(rr['codes'][:]==rc.index(region)))
    def h5_strings(obj,rows):
        if isinstance(obj,h5py.Group):
            cat=np.array([decode(v) for v in obj['categories'][:]],dtype=object)
            codes=np.asarray(obj['codes'][rows]);return np.where(codes>=0,cat[np.maximum(codes,0)],'')
        return np.array([decode(v) for v in obj[rows]],dtype=object)
    def original_cell_ids(h,rows):
        o=h['obs'];return h5_strings(o[decode(o.attrs.get('_index','_index'))],rows)

    S={}
    for sid,(age,sample,region,expected,t) in SPECS.items():
        with h5py.File(DATA/f'{age.lower()}.h5ad','r') as h:
            rows=section_rows(h,sample,region)
            if len(rows)!=expected:raise RuntimeError(f'{age} stage count mismatch: {len(rows)} != {expected}')
            xy=np.asarray(h['obsm']['spatial'][rows,:2],np.float32)
            raw_ids=original_cell_ids(h,rows).astype(str)
        if not np.isfinite(xy).all():raise RuntimeError('Invalid raw spatial coordinates')
        S[sid]=dict(age=age,time=t,xy=xy,raw_ids=raw_ids,
                    ids=np.asarray([f's{sid:02d}_{age}:{v}' for v in raw_ids],dtype=str))

    # The integrated atlas has a different set of cells; use a conservative, local
    # nearest-neighbor match for H3 domain labels. This matches the old notebook's
    # original SOURCE_MATCH_RADIUS_FACTOR, but does not reuse its old model contours.
    with h5py.File(ATLAS,'r') as h:
        for sid,(_,sample,region,_,_) in SPECS.items():
            rows=section_rows(h,sample,region)
            xy=np.asarray(h['obsm']['spatial'][rows,:2],np.float32)
            H3=h5_strings(h['obs']['H3_annotation'],rows)
            hh=np.fromiter((str(v).startswith('EN-ET-SP-') for v in H3),bool,count=len(H3))
            tree=cKDTree(xy)
            subset=np.linspace(0,len(xy)-1,min(len(xy),2100),dtype=int)
            spacing=float(np.median(tree.query(xy[subset],k=6)[0][:,5]))
            S[sid].update(h3xy=xy[hh],atlas_tree=tree,atlas_h3=hh,spacing=spacing)
            print(f'{S[sid]["age"]}: {len(S[sid]["xy"]):,} observed cells; {hh.sum():,} H3 EN-ET-SP reference cells',flush=True)
        source_dist,source_neighbor=S[1]['atlas_tree'].query(S[1]['xy'],k=1,workers=-1)
        source_mask=(source_dist<=SOURCE_MATCH_RADIUS_FACTOR*S[1]['spacing']) & S[1]['atlas_h3'][source_neighbor]
    source_rows=np.flatnonzero(source_mask)
    if len(source_rows)<N_SV:raise RuntimeError(f'Only {len(source_rows)} source H3 matches; require {N_SV}')
    source_xy=S[1]['xy'][source_rows]
    cohort=pd.DataFrame({'cell_id':S[1]['ids'][source_rows],
                         'stage_row':source_rows.astype(np.int64),
                         'nearest_atlas_distance':source_dist[source_rows]})
    if COHORT.is_file():
        old_cohort=pd.read_csv(COHORT)
        if not np.array_equal(old_cohort.cell_id.astype(str),cohort.cell_id.astype(str)) or not np.array_equal(old_cohort.stage_row,cohort.stage_row):
            raise RuntimeError('Saved NEW-run H3 cohort drift; refusing to mix source IDs')
    else:cohort.to_csv(COHORT,index=False)
    sv_pick=np.sort(np.random.default_rng(SEED+929).choice(source_rows,size=N_SV,replace=False))
    sv_ids=S[1]['ids'][sv_pick]
    print(f'Newly derived GW15 H3 source cohort: {len(cohort):,}; stVCR subset {N_SV}',flush=True)

    # -------------------- stVCR checkpoint-specific native ODE export --------------
    if PHASE=='stvcr_export':
        # Run this phase in the original stVCR training environment. Zero retraining.
        import scanpy as sc
        for repo in [PROJECT/'stVCR/src',PROJECT/'stvcr/src',PROJECT/'stVCR',PROJECT/'stvcr']:
            if (repo/'stvcr'/'__init__.py').is_file():
                sys.path.insert(0,str(repo));break
        import stvcr
        from stvcr.training import model as _stv_model  # ensure pickle class registration
        prep=sc.read_h5ad(STV/'stvcr_preprocessed_3tissue.h5ad',backed='r')
        try:
            cols=set(prep.obsm)
            if not {'X_spatial_input','X_gene_input'}<=cols:raise RuntimeError('Missing stVCR official 2D/10D inputs')
            if 'original_slice_id' not in prep.obs:raise RuntimeError('No original_slice_id in stVCR preprocessing')
            pi=np.flatnonzero(prep.obs.original_slice_id.to_numpy()==1)
            if len(pi)!=410492:raise RuntimeError(f'stVCR GW15 n={len(pi)}, expected 410492')
            expected=np.array([f'1|{cid}' for cid in S[1]['raw_ids']],dtype=str)
            if not np.array_equal(prep.obs_names.to_numpy()[pi].astype(str),expected):
                raise RuntimeError('stVCR preprocessing cells differ from raw GW15 order; refusing false trajectories')
            pos=pi[sv_pick]
            xinternal=np.asarray(prep.obsm['X_spatial_input'][pos],np.float32)
            emb=np.asarray(prep.obsm['X_gene_input'][pos],np.float32)
            if emb.shape!=(N_SV,10) or xinternal.shape!=(N_SV,2):raise RuntimeError('Bad stVCR input shape')
        finally:prep.file.close()
        # Convert internal coordinates to physical coordinates using only the
        # GW15 training source pairs. Do NOT fit any transform to heldout H3 labels.
        affine=np.linalg.lstsq(np.column_stack([xinternal,np.ones(N_SV)]),S[1]['xy'][sv_pick],rcond=None)[0]
        fit=np.column_stack([xinternal,np.ones(N_SV)])@affine
        diameter=float(np.linalg.norm(np.ptp(S[1]['xy'],axis=0)))
        res=np.linalg.norm(fit-S[1]['xy'][sv_pick],axis=1)
        print('stVCR native->raw source affine max/median discrepancy:',res.max(),np.median(res),flush=True)
        if np.quantile(res,.99)>0.001*diameter:
            raise RuntimeError('stVCR spatial preprocessing is not affinely convertible to source native XY: inspect alignment/scale')
        net=torch.load(STV/'stvcr_model_full.pt',map_location=DEVICE,weights_only=False)
        rigid=torch.load(STV/'stvcr_rigid_transform.pt',map_location=DEVICE,weights_only=False)
        net.eval();rigid.eval()
        if not hasattr(net,'spatial_velocity_net') or not hasattr(net,'gene_velocity_net'):
            raise RuntimeError('stVCR checkpoint has unexpected dynamics model API')
        if tuple(rigid.translation.shape)!=(2,2):raise RuntimeError('Unexpected stVCR rigid transform shape')
        torch.set_num_threads(min(8,torch.get_num_threads()))
        xy_results={2:np.empty((N_SV,2),np.float32),7:np.empty((N_SV,2),np.float32)}
        def solve_segment(y,a,b,steps):
            h=(b-a)/steps
            def vel(state,tt):
                t=torch.tensor(tt,dtype=state.dtype,device=state.device)
                return torch.cat((net.spatial_velocity_net(t,state),net.gene_velocity_net(t,state)),dim=1)
            for k in range(steps):
                t=a+k*h
                k1=vel(y,t);k2=vel(y+k1*(h/2),t+h/2)
                k3=vel(y+k2*(h/2),t+h/2);k4=vel(y+k3*h,t+h)
                y=y+(k1+2*k2+2*k3+k4)*(h/6)
            return y
        with torch.inference_mode():
            for a in range(0,N_SV,512):
                b=min(N_SV,a+512)
                y=torch.as_tensor(np.column_stack([xinternal[a:b],emb[a:b]]),dtype=torch.float32,device=DEVICE)
                y=solve_segment(y,0.,5/7,184)
                out20=y[:,:2]@rigid.get_rot_matrix(0)+rigid.translation[0]
                xy_results[2][a:b]=(np.column_stack([out20.cpu().numpy(),np.ones(b-a)])@affine)
                y=solve_segment(y,5/7,1.,80)
                out22=y[:,:2]@rigid.get_rot_matrix(1)+rigid.translation[1]
                xy_results[7][a:b]=(np.column_stack([out22.cpu().numpy(),np.ones(b-a)])@affine)
                print(f'New stVCR checkpoint ODE: {b:,}/{N_SV:,}',flush=True)
        for sid in (2,7):
            if not np.isfinite(xy_results[sid]).all():raise RuntimeError('Nonfinite new stVCR spatial endpoints')
        rec={'run':str(STV),'model_sha256':sha(STV/'stvcr_model_full.pt'),
             'rigid_sha256':sha(STV/'stvcr_rigid_transform.pt'),
             'preprocessed_path':str(STV/'stvcr_preprocessed_3tissue.h5ad'),
             'n_source':N_SV,'source_ids_sha256':hashlib.sha256('\n'.join(sv_ids).encode()).hexdigest(),
             'source_affine':affine.tolist(), 'source_affine_p99_residual':float(np.quantile(res,.99)),
             'method':'RK4 native spatial+expression velocities 0->5/7->1; rigid map for target age',
             'growth_weighting':'not applied to individual source-cell path',
             'checkpoints_retrained':False}
        if STV_RECEIPT.exists():
            prior=json.loads(STV_RECEIPT.read_text())
            if prior['model_sha256']!=rec['model_sha256'] or prior['source_ids_sha256']!=rec['source_ids_sha256']:
                raise RuntimeError('Old stVCR export uses a different checkpoint/cohort; refuse overwrite')
        for sid in (2,7):
            tab=pd.DataFrame({'source_cell_id':sv_ids,'x_pred_native':xy_results[sid][:,0],
                              'y_pred_native':xy_results[sid][:,1]})
            if STV_CSV[sid].is_file():
                prior=pd.read_csv(STV_CSV[sid])
                if not np.array_equal(prior.source_cell_id.astype(str),sv_ids):
                    raise RuntimeError('Saved stVCR endpoints have different source identities')
                if not np.allclose(prior[['x_pred_native','y_pred_native']],tab[['x_pred_native','y_pred_native']],atol=1e-4):
                    raise RuntimeError('stVCR regeneration changed endpoints; refuse overwrite')
            else:tab.to_csv(STV_CSV[sid],index=False)
        STV_RECEIPT.write_text(json.dumps(rec,indent=2))
        print('STVCR EXPORT COMPLETE; now set QIAN_BARY_PHASE=figure in Spatial TI kernel.')
    else:
        # Read only endpoints CREATED FROM THE NEW stVCR three-age checkpoint.
        if not STV_RECEIPT.is_file() or any(not p.is_file() for p in STV_CSV.values()):
            raise FileNotFoundError('NEW stVCR endpoints not exported. In the official stVCR kernel set QIAN_BARY_PHASE=stvcr_export and run this cell first.')
        strec=json.loads(STV_RECEIPT.read_text())
        if strec['model_sha256']!=sha(STV/'stvcr_model_full.pt'):
            raise RuntimeError('The new stVCR checkpoint changed since endpoint export')
        if strec['source_ids_sha256']!=hashlib.sha256('\n'.join(sv_ids).encode()).hexdigest():
            raise RuntimeError('New stVCR endpoint cohort is stale')
        sv20,sv22=pd.read_csv(STV_CSV[2]),pd.read_csv(STV_CSV[7])
        for tab in (sv20,sv22):
            if not np.array_equal(tab.source_cell_id.astype(str),sv_ids):
                raise RuntimeError('New stVCR endpoint source ordering mismatch')
            if not np.isfinite(tab[['x_pred_native','y_pred_native']].to_numpy()).all():
                raise RuntimeError('Nonfinite stVCR endpoint CSV')
        print('PASS new trained stVCR endpoints for exact 6,000 source identities',flush=True)
        # ---------------- new MOSCOT 0.5.2 saved factor/order audit ------------------
        # Version-specific trainer output: GW15_to_GW20 and GW20_to_GW22,
        # factor rows correspond to time-selected rows in moscot_input_3sections.h5ad.
        MOS_PAIRS={(1,2):'GW15_to_GW20',(2,7):'GW20_to_GW22'}
        MOS_OFFSETS={1:0,2:410492,7:410492+665204}
        with h5py.File(MOS/'moscot_input_3sections.h5ad','r') as h:
            ds=h['obs'][decode(h['obs'].attrs.get('_index','_index'))]
            expected_total=410492+665204+439986
            if len(ds)!=expected_total:raise RuntimeError('Wrong moscot three-age AnnData size')
            for sid in SPECS:
                offset=MOS_OFFSETS[sid]
                expected=S[sid]['raw_ids']
                # Check ALL saved observation IDs in chunks rather than only first/last.
                for a in range(0,len(expected),25000):
                    b=min(len(expected),a+25000)
                    observed=h5_strings(ds,slice(offset+a,offset+b)).astype(str)
                    assert np.array_equal(observed,np.asarray([f's{sid}:{v}' for v in expected[a:b]],dtype=str)), \
                        f'Moscot input observation ordering does not match raw section {sid}, rows {a}:{b}'
        print('PASS NEW moscot AnnData full cell-ID ordering against the raw 3 source tissues',flush=True)

        def frozen_factors(a,b):
            folder=MOS/MOS_PAIRS[(a,b)]
            Q=np.load(folder/'Q.npy',mmap_mode='r');R=np.load(folder/'R.npy',mmap_mode='r')
            g=np.asarray(np.load(folder/'g.npy'),dtype=np.float64).reshape(-1)
            assert Q.shape==(len(S[a]['xy']),200) and R.shape==(len(S[b]['xy']),200) and g.shape==(200,)
            assert (g>0).all() and np.isfinite(g).all()
            src=np.asarray(np.load(folder/'source_obs_positions.npy'),dtype=np.int64)
            dst=np.asarray(np.load(folder/'target_obs_positions.npy'),dtype=np.int64)
            assert np.array_equal(src,MOS_OFFSETS[a]+np.arange(len(S[a]['xy']),dtype=np.int64))
            assert np.array_equal(dst,MOS_OFFSETS[b]+np.arange(len(S[b]['xy']),dtype=np.int64))
            info=json.loads((folder/'solution_manifest.json').read_text())
            if info.get('factor_formula')!='T = Q @ diag(1/g) @ R.T':raise RuntimeError('Unexpected moscot OT factor formula')
            for fn in ('Q.npy','R.npy','g.npy'):
                if 'sha256' in info and sha(folder/fn)!=info['sha256'][fn]:raise RuntimeError('MOSCOT factor hash drift '+fn)
            print(f'PASS NEW moscot {a}->{b}: source/target IDs, positions, rank 200, hashes',flush=True)
            return Q,R,g

        def right_moments(R,features,block=BARY_MOMENT_BLOCK):
            features=np.asarray(features,dtype=np.float64)
            assert features.ndim==2 and len(features)==len(R)
            result=np.zeros((R.shape[1],features.shape[1]),dtype=np.float64)
            for lo in range(0,len(R),block):
                hi=min(lo+block,len(R))
                result+=np.asarray(R[lo:hi],dtype=np.float64).T @ features[lo:hi]
            return result

        def conditional_projection(Q,g,moments,rows=None,block=BARY_Q_BLOCK):
            # Last column of moments represents conditional row masses.
            if rows is None:
                rows=np.arange(len(Q),dtype=np.int64)
            rows=np.asarray(rows,dtype=np.int64)
            result=np.empty((len(rows),moments.shape[1]-1),dtype=np.float64)
            masses=np.empty(len(rows),dtype=np.float64)
            coeff=np.asarray(moments,dtype=np.float64)/g[:,None]
            for lo in range(0,len(rows),block):
                hi=min(lo+block,len(rows))
                vals=np.asarray(Q[rows[lo:hi]],dtype=np.float64) @ coeff
                denom=vals[:,-1]
                if not np.isfinite(vals).all() or np.any(denom<=0):
                    raise RuntimeError(f'Invalid OT row masses at rows [{lo}:{hi}]')
                result[lo:hi]=vals[:,:-1]/denom[:,None]
                masses[lo:hi]=denom
            if not np.isfinite(result).all():raise RuntimeError('Nonfinite barycenters')
            return result,masses

        # PROVE the low-rank composed conditional expectation using independent dense
        # matrix multiplication on a small deterministic synthetic OT fixture.
        _r=np.random.default_rng(20261029)
        _Q12=_r.uniform(.1,1.,(7,4));_R12=_r.uniform(.1,1.,(9,4));_g12=_r.uniform(.1,1.,4)
        _Q27=_r.uniform(.1,1.,(9,3));_R27=_r.uniform(.1,1.,(6,3));_g27=_r.uniform(.1,1.,3)
        _X20=_r.normal(size=(9,2));_X22=_r.normal(size=(6,2))
        _P12=(_Q12/_g12)@_R12.T;_P27=(_Q27/_g27)@_R27.T
        _K12=_P12/_P12.sum(1,keepdims=True);_K27=_P27/_P27.sum(1,keepdims=True)
        _ones=np.ones((6,1))
        _toy27,_=conditional_projection(_Q27,_g27,right_moments(_R27,np.column_stack((_X22,_ones))))
        _toyboth,_=conditional_projection(_Q12,_g12,right_moments(_R12,np.column_stack((_X20,_toy27,np.ones((9,1))))))
        assert np.allclose(_toy27,_K27@_X22,rtol=1e-11,atol=1e-11)
        assert np.allclose(_toyboth[:,:2],_K12@_X20,rtol=1e-11,atol=1e-11)
        assert np.allclose(_toyboth[:,2:],_K12@(_K27@_X22),rtol=1e-11,atol=1e-11)
        print('PASS exact low-rank two-hop conditional-barycenter formula (dense toy).',flush=True)
        del _Q12,_R12,_g12,_Q27,_R27,_g27,_P12,_P27,_K12,_K27,_X20,_X22,_toy27,_toyboth,_ones


        Q27,R27,g27=frozen_factors(2,7)
        moment27=right_moments(R27,np.column_stack((S[7]['xy'],np.ones(len(S[7]['xy'])))))
        # Important: all 665,204 intermediate-cell barycenters are required to compose.
        mid_to_22,mass27=conditional_projection(Q27,g27,moment27)
        print('Computed GW20->GW22 barycenters for',len(mid_to_22),'GW20 cells; '
              'min/max OT row mass',np.min(mass27),np.max(mass27),flush=True)
        del Q27,R27,g27,moment27,mass27

        Q12,R12,g12=frozen_factors(1,2)
        moment12=right_moments(R12,np.column_stack((S[2]['xy'],mid_to_22,
                                                   np.ones(len(S[2]['xy'])))))
        both,mass12=conditional_projection(Q12,g12,moment12,rows=source_rows)
        mos_xy={2:both[:,:2].copy(),7:both[:,2:4].copy()}
        print('Computed exact composed barycentric endpoints for',len(source_rows),'source cells.',flush=True)

        # Numerical spot-check against a DIRECT dense row of the huge OT plan,
        # comparing GW20 and GW22 barycenters, without ever assembling all rows.
        for rr in np.unique(np.linspace(0,len(source_rows)-1,3,dtype=int)):
            src=int(source_rows[rr])
            # Independently sum the full target weights in BLOCKS (no massive dense P).
            source_coeff=np.asarray(Q12[src],dtype=np.float64)/g12
            total=0.0;sum20=np.zeros(2);sum22=np.zeros(2)
            for lo in range(0,len(R12),BARY_MOMENT_BLOCK):
                hi=min(lo+BARY_MOMENT_BLOCK,len(R12))
                w=np.asarray(R12[lo:hi],dtype=np.float64)@source_coeff
                if not np.isfinite(w).all() or w.min() < -1e-10:
                    raise RuntimeError('Direct coupling check found invalid conditional weights')
                total+=float(w.sum())
                sum20+=w@S[2]['xy'][lo:hi]
                sum22+=w@mid_to_22[lo:hi]
            assert total>0
            assert np.allclose(sum20/total,mos_xy[2][rr],rtol=1e-7,atol=1e-6)
            assert np.allclose(sum22/total,mos_xy[7][rr],rtol=1e-7,atol=1e-6)
        print('PASS direct full-destination row verification of both age endpoints.',flush=True)
        del Q12,R12,g12,moment12,mid_to_22,both,mass12

        # Spatial validity diagnostic: an expectation of points in a crescent can lie
        # between tissue arms. Quantify nearest actual target-cell separation, but do
        # NOT project/snap barycenters to tissue, since that changes the OT estimate.
        qc=[]
        for sid in (2,7):
            pred=mos_xy[sid]; target=S[sid]['xy']
            stage_lo=target.min(0);stage_hi=target.max(0)
            outside_box=np.any((pred<stage_lo)|(pred>stage_hi),axis=1)
            d=cKDTree(target).query(pred,k=1,workers=-1)[0]
            threshold=3*S[sid]['spacing']
            diagnostics={'target':S[sid]['age'],'n_sources':len(pred),
                         'mean_nearest_tissue_distance':float(d.mean()),
                         'median_nearest_tissue_distance':float(np.median(d)),
                         'p95_nearest_tissue_distance':float(np.quantile(d,.95)),
                         'distance_threshold_3x_atlas_spacing':float(threshold),
                         'fraction_farther_than_3x_spacing':float((d>threshold).mean()),
                         'fraction_outside_target_bbox':float(outside_box.mean()),
                         'fraction_finite':float(np.isfinite(pred).all(axis=1).mean())}
            qc.append(diagnostics)
            print(S[sid]['age'],'barycenter tissue-support QC:',diagnostics,flush=True)
            if diagnostics['fraction_outside_target_bbox']>1e-6:
                raise RuntimeError('Barycenters outside target bounding box; check coupling weights')
        qc=pd.DataFrame(qc)
        qc.to_csv(OUT/'moscot_barycenter_tissue_support_diagnostics.csv',index=False)
        mos=pd.DataFrame({'gw15_cell_id':cohort.cell_id.astype(str).to_numpy(),
                          'gw15_stage_row':source_rows,
                          'gw15_x':source_xy[:,0],'gw15_y':source_xy[:,1],
                          'gw20_x':mos_xy[2][:,0],'gw20_y':mos_xy[2][:,1],
                          'gw22_x':mos_xy[7][:,0],'gw22_y':mos_xy[7][:,1]})
        mos.to_csv(OUT/'ENETSP_GW15_to_GW20_GW22_BARYCENTRIC_EXPECTED_PATHS.csv.gz',index=False)
        print('Saved exact barycentric endpoints; target H3 labels never used to compute them.',flush=True)